# Financial Trading with Python, 2nd Edition

Cordell L. Tanny, CFA, FRM, FDP

**Chapter 16: AI (LLMs) as an Alpha Signal Generator**

Notebook 16.1: Building a Point-in-Time FOMC Corpus

Version: 1.1

Last revised: 2026-08-22

**Optional. This notebook shows what it costs to assemble a text corpus clean enough to feed a model, using thirty years of Federal Reserve policy statements as the example.**

> Save a copy to your own Drive before running: File > Save a copy in Drive. Edits to the shared notebook are not preserved.

> This notebook is long, and none of it is required. Notebook 16.2 is fully independent: it loads the finished corpus (`fomc_statements.csv`) from the book's repository and never runs any of the code below. Read this one if you want to see what text acquisition actually involves. Skip it if you came for the analysis.

## 1.0 Introduction

Before anything else: you do not need this notebook.

- **If you came for the analysis, skip to Notebook 16.2.** It loads
  `fomc_statements.csv` from the book's repository and depends on nothing here.
  No variable, function, or file produced below is imported there.
- **If you want to see what text acquisition costs, read on.** Running it is
  optional. The failures below read perfectly well without a Colab session
  attached.
- **If you plan to scrape a different document set, run it.** The structure
  transfers even though the website will not.

Everything that follows exists because of one asymmetry. Feeding text to a model,
even a very basic agentic setup like the one in 16.2, is the easy part. Getting
text worth feeding it is not, and nothing in the process tells you when you have
failed.

- Ask yfinance for SPY prices and the DataFrame's shape tells you at once whether
  you got what you asked for.
- A document scrape gives you whatever your extractor caught, and nothing in the
  output announces what it missed.

So every check here comes from outside the code: how many times the FOMC meets in
a year, how long a policy statement runs, the fact that the Committee records its
votes. Your parser cannot fake any of those.

The notebook is built as a sequence of four failures rather than as finished
working code, because three of them produced output that looked entirely
reasonable and the fourth produced a research finding that was not real. That is
the demonstration. The corpus is only the by-product.

**What it produces:** `fomc_statements.csv`, every FOMC policy statement from
1994 forward, with the voting roster split off and provenance stamped on each
row.

## 2.0 Setup: Imports and Configuration

Where does every setting in this notebook live? Here, in one cell, so you never
have to hunt for a constant you want to change.

Two settings deserve a word before you run anything:

- **`REQUEST_DELAY` is an obligation, not a preference.** Thousands of readers
  will run this against a public agency's web server. One second between requests
  costs you a few minutes and costs the Fed nothing. Please do not lower it, and
  please put a real contact address in `USER_AGENT` so anyone reading the server
  logs can tell what the traffic is.
- **`CORPUS_END_DATE` is what makes this reproducible.** The Fed keeps publishing,
  so a run in 2027 returns a larger corpus and none of the chapter's numbers
  reconcile. Move it forward deliberately, not by accident.

No regex patterns appear here. Those are built in Sections 4.0 through 6.0, one
failure at a time, and putting the finished versions at the top would give away
the point of those sections.

What to look for in the output: nothing yet. Confirm the version numbers print
and the cache directory resolves to somewhere you can write.

In [1]:
# =============================================================================
# Section 2.0: Configuration
# Every setting used anywhere in this notebook, centralized.
# =============================================================================

import re
import time
import unicodedata
from datetime import date
from pathlib import Path
import json
from datetime import datetime, timezone
from urllib.parse import urljoin

import bs4
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

# --- Source pages ------------------------------------------------------------
BASE_URL = "https://www.federalreserve.gov"

# We parse the Fed's own calendar pages instead of constructing URLs from
# meeting dates. Meeting dates move, unscheduled actions exist, and a URL we
# built ourselves that returns 404 is indistinguishable from a meeting that
# never happened. The calendar pages also preserve whichever URL scheme was
# live at the time, which matters because there have been three of them.
CALENDAR_CURRENT = f"{BASE_URL}/monetarypolicy/fomccalendars.htm"
CALENDAR_HISTORICAL_TEMPLATE = BASE_URL + "/monetarypolicy/fomchistorical{year}.htm"

# Historical pages exist through 2020. Years from 2021 forward live on the
# current calendar page only, so requesting a historical page for those years
# returns a 404 that we expect and treat as an answer rather than an error.
HISTORICAL_YEARS = range(1994, 2021)

# --- Corpus window -----------------------------------------------------------
# Starting in 1994 is deliberate. The FOMC first announced a meeting outcome in
# February 1994 and did not commit to a statement after every scheduled meeting
# until January 2000, so 1994 through 1996 should come back empty. Sweeping
# those years anyway turns three zeros into evidence that the scraper looked,
# rather than three years that are missing because we never asked.
CORPUS_START_DATE = "1994-01-01"

# Pinned so that a reader's run reproduces the numbers printed in the chapter.
CORPUS_END_DATE = "2026-07-31"

# --- Politeness --------------------------------------------------------------
REQUEST_DELAY = 1.0        # seconds between requests, an obligation not a setting
REQUEST_TIMEOUT = 30       # seconds before a hung connection is abandoned
MAX_RETRIES = 2            # applies to timeouts and 5xx only, never to a 404

# Replace the contact address with your own before running.
USER_AGENT = (
    "FinancialTradingWithPython-2e/16.1 research script "
    "(Packt B37124; contact: your.email@example.com)"
)
REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# --- Decoding ----------------------------------------------------------------
# Tried in order against the raw bytes. We do not trust the declared charset:
# older Fed pages report iso-8859-1 while serving UTF-8, which is Section 7.0.
DECODE_ENCODINGS = ("utf-8", "latin-1")

# BeautifulSoup selects a parser based on what is installed, which means two
# readers can get different paragraph trees from identical bytes. Legacy Fed
# pages are table-layout HTML with unclosed tags, and parsers repair that
# differently. html.parser ships with Python, so this is the one choice that
# does not depend on the environment.
HTML_PARSER = "html.parser"

# --- Extraction guards (used in Section 8.0) -----------------------------------
MIN_PARAGRAPH_CHARS = 40    # shorter paragraphs are navigation and furniture
MAX_ANCHOR_TEXT_RATIO = 0.60  # a paragraph mostly made of links is a link list

# --- Storage -----------------------------------------------------------------
# Colab storage is wiped when the runtime disconnects. If you plan to iterate,
# mount Drive and point CACHE_DIR at a folder there so a second run reuses the
# HTML you already fetched instead of hitting the Fed again.
CACHE_DIR = Path("fomc_cache")
RAW_HTML_DIR = CACHE_DIR / "raw_html"
OUTPUT_CSV = CACHE_DIR / "fomc_statements.csv"
USE_CACHE = True            # set False to force a re-fetch of every page

RAW_HTML_DIR.mkdir(parents=True, exist_ok=True)

# --- Provenance --------------------------------------------------------------
# Stamped onto every row of the output. A corpus without a scrape date and a
# script version cannot be reconciled against a later run.
SCRIPT_VERSION = "16.1.0"
SCRAPE_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")  # Colab VMs run UTC

# --- Confirm the environment -------------------------------------------------
print(f"pandas {pd.__version__} | numpy {np.__version__} | "
      f"requests {requests.__version__} | beautifulsoup4 {bs4.__version__}")
print(f"Corpus window   : {CORPUS_START_DATE} to {CORPUS_END_DATE}")
print(f"Cache directory : {CACHE_DIR.resolve()}")
print(f"Scrape date     : {SCRAPE_DATE} UTC (stamped into every output row)")
print(f"Request delay   : {REQUEST_DELAY:.1f}s")

pandas 2.2.3 | numpy 2.1.3 | requests 2.32.4 | beautifulsoup4 4.13.5
Corpus window   : 1994-01-01 to 2026-07-31
Cache directory : /content/fomc_cache
Scrape date     : 2026-08-22 UTC (stamped into every output row)
Request delay   : 1.0s


## 3.0 Why Text Has No Completeness Check

How do you know you got all the data?

Price data answers that by itself:

- Ask yfinance for SPY from 2000 and you get a DataFrame with a shape.
- Wrong row count, wrong start date, gaps in the middle: all visible before you
  write a line of analysis.

Text gives you no such thing:

- A scrape returns what your code caught, not what exists.
- A document you never found leaves no row behind, so there is no gap to see.
- Output from a broken scraper and from a working one look identical.

The check has to come from outside the code, meaning a fact about the world your
scraper had no part in producing. This notebook uses three:

| Check | The outside fact | What it catches |
| --- | --- | --- |
| Meeting cadence | The FOMC holds eight scheduled meetings a year, and has issued a statement after every one since January 2000 | Documents you never found |
| Document length | A policy statement runs a few hundred words, not a few thousand | Extra text scooped up along with the statement |
| Voting record | Every statement records how the members voted | A parser that quietly stopped matching |

Each failure below was caught by one of these. None was caught by the code.

Worth saying plainly: I did not write the checks first and then go find the bugs.
I built the scraper, it produced sensible looking output, and I went looking only
because a number in a print statement disagreed with something I already knew.

## 4.0 Discovering Statement URLs

Where does the list of statements come from? Two ways:

- **Build the URLs yourself from meeting dates.** This fails quietly, because a
  URL you invented that returns 404 looks exactly like a meeting that never
  happened. Dates also move, and emergency actions are on no calendar you can
  guess.
- **Read the Fed's own calendar pages and take the links off them.** We then look
  for the statements the Fed says exist rather than the ones we assumed.

We take the second. The code does three things:

1. Fetches the calendar pages, one for recent years and one per historical year.
2. Collects every link on those pages.
3. Keeps the links that look like a policy statement.

Two choices worth knowing about now:

- **Cache raw bytes, not decoded text.** Section 7.0 finds a decoding bug, and
  caching bytes means we fix it from disk instead of re-fetching every page.
- **Treat 404 as an answer, not a reason to retry.** Historical calendar pages
  exist only through 2020, so asking for 2021 onward returns 404 by design.

Step 3 uses the obvious patterns: one for modern URLs, one for the pre-2006
scheme. Two schemes, two patterns, and it looks complete.

What to look for in the output:

- Ignore the total. It will look fine.
- Read the per-year counts against eight meetings a year.
- Runtime is about thirty seconds for twenty eight pages.

This is going to be a little overwhelming. I know there is a lot of code and functions, but you need to understand the entire process.

In [2]:
# =============================================================================
# Section 4.0: Discovering statement URLs, attempt one
# =============================================================================

CORPUS_START = pd.Timestamp(CORPUS_START_DATE)
CORPUS_END = pd.Timestamp(CORPUS_END_DATE)


# --- The fetcher, used by every section from here on -------------------------

def _cache_paths(url):
    """Map a URL to its cached bytes file and its metadata sidecar."""
    slug = re.sub(r"[^A-Za-z0-9]+", "_", url.replace(BASE_URL, "")).strip("_")
    return RAW_HTML_DIR / f"{slug}.html", RAW_HTML_DIR / f"{slug}.json"


def fetch_raw(url, session):
    """
    Return (raw_bytes, meta) for one page, or (None, meta) when it is not there.

    We cache the undecoded bytes and the charset the server claimed, rather than
    caching a decoded string. If we later discover we decoded it wrong, and in
    Section 7.0 we do, the correction runs against the cache instead of against
    the Fed's servers.
    """
    body_path, meta_path = _cache_paths(url)

    if USE_CACHE and body_path.exists() and meta_path.exists():
        return body_path.read_bytes(), json.loads(meta_path.read_text())

    for attempt in range(MAX_RETRIES + 1):
        try:
            response = session.get(url, timeout=REQUEST_TIMEOUT)
        except requests.RequestException:
            time.sleep(REQUEST_DELAY * (attempt + 1))
            continue

        # A 404 is information, not an error to retry. The page is not there and
        # asking a second time will not change that.
        if response.status_code == 404:
            return None, {"url": url, "status": 404, "fetched_at": SCRAPE_DATE}

        if response.status_code >= 500:
            time.sleep(REQUEST_DELAY * (attempt + 1))
            continue

        response.raise_for_status()
        meta = {
            "url": url,
            "status": response.status_code,
            "declared_encoding": response.encoding,
            "fetched_at": SCRAPE_DATE,
        }
        body_path.write_bytes(response.content)
        meta_path.write_text(json.dumps(meta))
        time.sleep(REQUEST_DELAY)
        return response.content, meta

    return None, {"url": url, "status": "unreachable", "fetched_at": SCRAPE_DATE}


def fetch_html(url, session):
    """
    Decode a page using the charset the server declared.

    Trusting the declaration is the obvious thing to do, and on this site it is
    wrong. Section 7.0 shows where it breaks and replaces this function.
    """
    raw, meta = fetch_raw(url, session)
    if raw is None:
        return None
    return raw.decode(meta.get("declared_encoding") or "utf-8", errors="replace")


# --- Reading the Fed's calendar pages ----------------------------------------

def collect_calendar_html(session):
    """Fetch every calendar page that could list a policy statement."""
    pages = {}

    html = fetch_html(CALENDAR_CURRENT, session)
    if html is not None:
        pages[CALENDAR_CURRENT] = html

    for year in HISTORICAL_YEARS:
        url = CALENDAR_HISTORICAL_TEMPLATE.format(year=year)
        html = fetch_html(url, session)
        if html is None:
            print(f"  {year}: no historical calendar page")
            continue
        pages[url] = html

    return pages


def links_on_page(html, page_url):
    """Every link on a page as an absolute URL, with fragments stripped."""
    soup = BeautifulSoup(html, HTML_PARSER)
    return {
        urljoin(page_url, anchor["href"]).split("#")[0]
        for anchor in soup.find_all("a", href=True)
    }


# --- Attempt one -------------------------------------------------------------
# Modern statements sit under /newsevents/pressreleases/ with the date glued to
# a "monetary" prefix. Pre-2006 statements sit under /boarddocs/ in a folder
# named for the date. Two schemes, two patterns, and that looks like all of it.
RE_MODERN_ATTEMPT1 = re.compile(r"/newsevents/pressreleases/monetary(\d{8})a\.htm$")
RE_LEGACY = re.compile(r"/boarddocs/press/(?:monetary|general)/\d{4}/(\d{8})/?(?:default\.htm)?$")


def statement_urls(pages, patterns):
    """
    Return {statement_date: url} for links matching any of the supplied patterns.

    Keying on the date deduplicates the same statement appearing on more than
    one calendar page, which happens at the boundary between the current and
    historical listings.
    """
    found = {}
    for page_url, html in pages.items():
        for link in links_on_page(html, page_url):
            for pattern in patterns:
                match = pattern.search(link)
                if match:
                    stamp = pd.to_datetime(match.group(1), format="%Y%m%d")
                    if CORPUS_START <= stamp <= CORPUS_END:
                        found[stamp] = link
                    break
    return dict(sorted(found.items()))


session = requests.Session()
session.headers.update(REQUEST_HEADERS)

print("Fetching calendar pages")
calendar_pages = collect_calendar_html(session)
print(f"\nCalendar pages retrieved: {len(calendar_pages)}")

urls_attempt1 = statement_urls(calendar_pages, [RE_MODERN_ATTEMPT1, RE_LEGACY])
print(f"Statement URLs found: {len(urls_attempt1)}\n")

year_counts = (
    pd.Series([stamp.year for stamp in urls_attempt1])
    .value_counts()
    .reindex(range(CORPUS_START.year, CORPUS_END.year + 1), fill_value=0)
    .sort_index()
)

print("Statements per year")
for year, count in year_counts.items():
    print(f"  {year}: {count}")

Fetching calendar pages

Calendar pages retrieved: 28
Statement URLs found: 192

Statements per year
  1994: 0
  1995: 0
  1996: 0
  1997: 1
  1998: 3
  1999: 6
  2000: 8
  2001: 11
  2002: 8
  2003: 8
  2004: 8
  2005: 8
  2006: 0
  2007: 0
  2008: 0
  2009: 0
  2010: 0
  2011: 8
  2012: 8
  2013: 8
  2014: 8
  2015: 8
  2016: 8
  2017: 8
  2018: 8
  2019: 9
  2020: 12
  2021: 8
  2022: 8
  2023: 8
  2024: 8
  2025: 9
  2026: 5


Read the total first: 192 statements across thirty years. A believable number,
and if it were the only thing printed we would move on.

Now read the years:

| Years | Count | Verdict |
| --- | --- | --- |
| 1994 to 1996 | 0 | Correct. The FOMC did not issue statements in these years. |
| 1997 to 1999 | 1, 3, 6 | Correct. Statements issued only when policy changed. |
| 2000 onward | 8 in most years | Correct. This is the scheduled cadence. |
| 2006 to 2010 | 0 | Wrong. |

Five consecutive years of nothing, covering:

- Two full years of rate increases into 2006
- The credit crunch beginning in 2007
- The failure of Lehman Brothers
- The cut to near zero in December 2008
- The first two rounds of quantitative easing

Roughly forty statements missing, including every statement made during the
financial crisis. Any research question about how the Fed communicates under
stress would be answered from a corpus with the stress removed.

What the code did about it:

- No exception, no warning, no empty result.
- Both regex patterns worked and every page fetched.
- The scraper did precisely what it was told and produced a file that reads as
  complete.

The zeros are visible only because of two earlier decisions:

- We printed statements per year rather than a total.
- We used `reindex(fill_value=0)`, so a year with no statements prints a zero
  instead of dropping out of the list entirely.

Without that second line there would be nothing to notice.

A few counts sit one above the cadence: 2019 has nine, 2020 twelve, 2025 nine.
Some of that is real, since the Committee meets outside its schedule during
emergencies, and some is not. Section 10.0 sorts out which.

First, the missing five years.

## 5.0 Finding the Addresses That Were Missed

Why did 2006 through 2010 come back empty? The statements are on the Fed's site
and the calendar pages link to them, so the pattern did not match.

The obvious fix, and I tried it first:

- Our pattern was anchored on the folder `/newsevents/pressreleases/`.
- The filename is what identifies the document.
- So drop the folder and match on the filename alone.

It returned the same 192 URLs. Not one statement recovered, and nothing in the
output to say why. The reason is a single character, and it makes more sense
after looking at the page than before.

Which is this section. Two guesses have been wrong, and a third costs another
cycle of writing a pattern, running the scrape, and reading the year counts.
Reading the addresses off the page costs a few seconds.

The method:

- Take one calendar page from the era that came back empty.
- Pull every link off it.
- Keep the ones containing an eight digit date, since any Fed address that
  identifies a meeting will have one.
- Group them by folder, so a long list of addresses becomes a short list of
  shapes.

What to look for in the output:

- Which folders hold dated documents in 2008.
- The example filename under each, compared against our two patterns.

In [3]:
# =============================================================================
# Diagnostic: what do the 2006 to 2010 addresses actually look like?
# =============================================================================

# Any year from the empty range works. 2008 has the most documents, which makes
# the folder structure easiest to see.
DIAGNOSTIC_YEAR = 2008
diagnostic_url = CALENDAR_HISTORICAL_TEMPLATE.format(year=DIAGNOSTIC_YEAR)

# The page is already in the cache from Section 4.0, so this costs nothing.
all_links = links_on_page(calendar_pages[diagnostic_url], diagnostic_url)
print(f"Links on the {DIAGNOSTIC_YEAR} calendar page: {len(all_links)}")

# Eight digits in a row. This is deliberately loose: we are not trying to
# identify statements yet, only to find every address that refers to a dated
# document. A loose pattern is the right tool when you are looking rather than
# selecting.
RE_ANY_DATE = re.compile(r"\d{8}")
dated_links = sorted(link for link in all_links if RE_ANY_DATE.search(link))
print(f"Links containing an eight digit date: {len(dated_links)}\n")

# Split each address at the last slash. Everything before it is the folder,
# everything after is the filename. Counting the folders collapses dozens of
# addresses into the handful of shapes the Fed actually uses.
folders = [link.rsplit("/", 1)[0].replace(BASE_URL, "") for link in dated_links]
folder_counts = pd.Series(folders).value_counts()

print(f"Folders holding dated documents in {DIAGNOSTIC_YEAR}")
for folder, count in folder_counts.items():
    # Pull one filename from this folder as a concrete example, since the
    # folder alone does not tell us what the files are named.
    example = next(
        link.rsplit("/", 1)[1]
        for link in dated_links
        if link.rsplit("/", 1)[0].replace(BASE_URL, "") == folder
    )
    print(f"  {folder}")
    print(f"      {count} files, for example: {example}")

Links on the 2008 calendar page: 330
Links containing an eight digit date: 134

Folders holding dated documents in 2008
  /monetarypolicy/files
      100 files, for example: FOMC20080109confcall.pdf
  /newsevents/press/monetary
      11 files, for example: 20080122b.htm
  /monetarypolicy
      8 files, for example: fomc20080625.htm
  /fomc/beigebook/2008/20081015
      2 files, for example: default.htm
  /fomc/beigebook/2008/20080416
      2 files, for example: default.htm
  /fomc/beigebook/2008/20080305
      2 files, for example: default.htm
  /fomc/beigebook/2008/20080611
      2 files, for example: default.htm
  /fomc/beigebook/2008/20080723
      2 files, for example: default.htm
  /fomc/beigebook/2008/20080903
      2 files, for example: default.htm
  /fomc/beigebook/2008/20081203
      2 files, for example: default.htm
  /fomc/beigebook/2008/20080116
      1 files, for example: default.htm


The 2008 page carries 330 links, of which 134 point at a dated document. Most
are not statements:

| Folder | Files | Example | What it holds |
| --- | --- | --- | --- |
| `/monetarypolicy/files` | 100 | `FOMC20080109confcall.pdf` | Transcripts and supporting PDFs |
| `/newsevents/press/monetary` | 11 | `20080122b.htm` | Press releases, statements among them |
| `/monetarypolicy` | 8 | `fomc20080625.htm` | Meeting pages |
| `/fomc/beigebook/2008/...` | 1 to 2 each | `default.htm` | Beige Book releases |

The second row is what we wanted, and the mismatch is one character:

- **What we searched for:** `monetary` followed immediately by eight digits, as
  in `monetary20240131a.htm`
- **What is there:** `monetary` as a folder name, then a slash, then eight
  digits, as in `/newsevents/press/monetary/20080122a.htm`

The word is present in both. In the modern format it is glued to the front of
the filename, and in the middle format it is the folder the file sits in. One
slash separates the two cases, and that slash is why five years of the crisis
went missing.

Two details to carry into the next section:

- **The example ends in `b.htm`, not `a.htm`.** The Fed publishes several
  documents per meeting under the same date with different letter suffixes, so
  the suffix does real work in the pattern. Section 6.0 tests this.
- **The Beige Book addresses are near misses.** They end in
  `/2008/20081015/default.htm`, close to the shape our pre-2006 pattern matches,
  and are rejected only because that pattern also requires `/boarddocs/press/` in
  front. A pattern loose enough to catch everything you want will usually catch
  things you do not.

### 5.1 The Pattern That Covers All Three Eras

The fix is to make the separator optional, so one pattern covers both formats:

- `/monetary` then no separator then the date, which is modern
- `/monetary` then a slash then the date, which is the middle era

That same optional character explains why matching the filename alone changed
nothing. `monetary(\d{8})a\.htm$` requires the word and the digits to sit next
to each other, true of modern addresses and false of the middle era, so removing
the folder left the actual mismatch untouched. The fix was reasonable, it
addressed a real weakness, and it was not the weakness causing the problem.

The pre-2006 pattern stays as it is, since it has been finding those years
correctly since attempt one.

What to look for in the output:

- The total against 192.
- 2006 through 2010, which should now hold eight to nine per year.
- Every other year, which should be unchanged. A fix that alters years it was
  not supposed to touch is not a fix.

In [4]:
# =============================================================================
# Section 5.0: One pattern for all three eras
# =============================================================================

# Comparing the two address formats side by side:
#
#   /newsevents/pressreleases/monetary20240131a.htm     modern, no separator
#   /newsevents/press/monetary/20080122a.htm            middle, one slash
#                            ^
#                            the only difference
#
# Reading the pattern piece by piece:
#
#   /monetary   a literal slash, then the word. The leading slash matters: it
#               forces "monetary" to be a whole path segment or the start of a
#               filename, so /monetarypolicy/... does not qualify.
#   /?          zero or one slash. The ? makes the character before it
#               optional, and this single character is the entire fix.
#   (\d{8})     eight digits, captured so we can read the date back out.
#   a           the Fed's suffix for the policy statement itself.
#   \.htm$      ends in .htm with nothing after it. The $ rejects a1.htm, the
#               implementation note, and the $ combined with the literal "a"
#               rejects b.htm and c.htm, which are separate documents.
RE_FILENAME = re.compile(r"/monetary/?(\d{8})a\.htm$")

# The intermediate attempt, kept because the test harness in Section 6.0 uses it.
# Dropping the folder and matching the filename alone looks like the right fix
# and returns exactly the same 192 URLs, because it still requires "monetary"
# and the date to be adjacent.
RE_FILENAME_ATTEMPT2 = re.compile(r"monetary(\d{8})a\.htm$")


def counts_by_year(urls):
    """
    Statements per year, with zeros kept in place.

    reindex with fill_value=0 is what makes this useful. Without it a year
    holding no statements is simply absent from the output and reads as
    nothing unusual.
    """
    return (
        pd.Series([stamp.year for stamp in urls])
        .value_counts()
        .reindex(range(CORPUS_START.year, CORPUS_END.year + 1), fill_value=0)
        .sort_index()
    )


# One pattern now covers 2006 to the present. The pre-2006 pattern is unchanged.
urls_attempt3 = statement_urls(calendar_pages, [RE_FILENAME, RE_LEGACY])

print(f"Attempt one   : {len(urls_attempt1)} URLs")
print(f"Attempt three : {len(urls_attempt3)} URLs\n")

comparison = pd.DataFrame({
    "attempt_1": counts_by_year(urls_attempt1),
    "attempt_3": counts_by_year(urls_attempt3),
})
comparison["change"] = comparison["attempt_3"] - comparison["attempt_1"]

print("Statements per year")
print(comparison.to_string())

# The years we were trying to repair, isolated so the result is not something
# you have to find in a thirty row table.
recovered = comparison.loc[2006:2010, "change"].sum()
print(f"\nStatements recovered from 2006 to 2010: {recovered}")

# The check that matters just as much: did the fix disturb anything else?
# Loosening a pattern to catch what it was missing usually widens it in other
# directions too, and those extra matches land in years that were correct.
other_years = comparison.drop(index=range(2006, 2011))
unexpected = other_years[other_years["change"] != 0]
if unexpected.empty:
    print("No change in any other year.")
else:
    print("\nYears changed that should not have been:")
    print(unexpected.to_string())


Attempt one   : 192 URLs
Attempt three : 235 URLs

Statements per year
      attempt_1  attempt_3  change
1994          0          0       0
1995          0          0       0
1996          0          0       0
1997          1          1       0
1998          3          3       0
1999          6          6       0
2000          8          8       0
2001         11         11       0
2002          8          8       0
2003          8          8       0
2004          8          8       0
2005          8          8       0
2006          0          8       8
2007          0          9       9
2008          0          9       9
2009          0          8       8
2010          0          9       9
2011          8          8       0
2012          8          8       0
2013          8          8       0
2014          8          8       0
2015          8          8       0
2016          8          8       0
2017          8          8       0
2018          8          8       0
2019          9    

The gap is closed: 235 URLs, 43 statements recovered from 2006 through 2010, no
other year touched.

The recovered counts read correctly against the cadence:

| Year | Count | Why it is not eight |
| --- | --- | --- |
| 2006 | 8 | Scheduled meetings only |
| 2007 | 9 | An intermeeting cut in August |
| 2008 | 9 | Intermeeting cuts and the coordinated action in October |
| 2009 | 8 | Scheduled meetings only |
| 2010 | 9 | Includes the coordinated swap line action in May |

Two checks ran on that fix and both were necessary:

- The years we meant to repair changed.
- No other year changed.

The second is easier to skip and the more common source of trouble. Had
`unexpected` come back with rows in it, the fix would have been wrong regardless
of the 43 statements it recovered.

The shape of the section, because it is what this work actually looks like:

1. Guessed the address format, was wrong, found out from the year counts.
2. Guessed the fix, was wrong, found out because the same check ran again.
3. Stopped guessing and read the addresses off the page.

One point to carry into the next section: this pattern is correct as far as we
know. That was also true of the first pattern, at the point when it had returned
192 plausible looking URLs.

## 6.0 Writing a Test Harness for a Pattern

Can we do better than running the whole scrape and reading the year counts?

Yes, and this is where the work starts paying you back. Three attempts have shown
what an address pattern gets wrong, so we write those cases down once with the
correct answer attached, and every future change to the pattern gets checked in
under a second instead of over a full scrape.

The harness holds two kinds of case:

- Addresses the pattern must match, at least one per era.
- Addresses it must reject, particularly the near misses.

The near misses are where the value is. The Fed publishes several documents per
meeting under one date, separated only by a letter:

| Address ends in | Document |
| --- | --- |
| `a.htm` | The policy statement, which is what we want |
| `a1.htm` | The implementation note |
| `b.htm` | Longer-run goals and strategy |
| `c.htm` | Balance sheet principles |

We add a decoy from a different Fed release family as well, so you can watch a
false positive get rejected rather than take my word for it.

Now the limit, stated up front so you know what you are buying:

- Every test case comes from a document you have already seen.
- So the harness confirms your pattern handles the cases you know about.
- My attempt one pattern would have passed a harness like this, twelve cases out
  of twelve, while missing the entire financial crisis.

That is not a flaw in the technique and no version of it fixes the problem,
because a harness cannot contain a case you have never encountered.

### 6.1 Using AI to Write and Check the Pattern

Regex is unpleasant to write and worse to read, and this is one of the places
where a language model genuinely helps. The help also has a sharp limit, and the
three failures above show exactly where it sits.

What a model does well:

- **Turning a description into a pattern.** "Match a URL ending in eight digits
  then a.htm, but not a1.htm" produces working syntax in one go.
- **Explaining a pattern someone else wrote.** Paste `/monetary/?(\d{8})a\.htm$`
  and ask what it matches, and the breakdown arrives faster than you would work
  it out.
- **Generating test cases.** Ask for near misses and you get a longer list than
  you would have thought of, which matters given how much of this section is
  about near misses.

What it does not do:

- **Tell you the Fed used three address schemes.** That is a fact about a
  specific website at a specific time, not something recoverable from the pattern
  you showed it.

Look back at the attempts. The first failed because the pattern was anchored on a
folder that had not always existed, and the second failed over one optional
slash. Neither is difficult regex, and a model would have written either pattern
correctly from my description. It would have written them correctly and they
would still have missed the crisis, because the description I gave was the wrong
description.

So ask a model for syntax and keep the question of what exists for yourself. In
practice:

- **Give it the actual addresses.** Paste twenty real URLs off the page and ask
  for a pattern that matches the statements and rejects the rest. Now it works
  from what is there rather than from what you assumed.
- **Ask it to find near misses to your pattern**, then check whether those
  documents exist on the site. This is the one direction where it can find a gap
  you did not know about.
- **Ask it to explain any pattern before you run it.** A pattern you cannot read
  is a pattern you cannot debug when the scrape looks wrong.
- **Never accept a pattern without running the harness.** The syntax will be
  right. Whether it matches the right documents is a separate question, and one
  the model has no way to answer.

### 6.2 Building the Harness

The harness is a list of cases, each an address paired with the expected answer
and a note saying why the case exists. The note matters: in six months you will
not remember why `a1.htm` was worth testing, and a case you cannot explain is a
case you will delete the next time it fails.

Fifteen cases in three groups:

- Five that must match, one per address scheme plus two variations.
- Eight near misses, all real documents the Fed publishes.
- Two decoys from other Fed release families.

Every address is real. Making them up would defeat the purpose, since the harness
works precisely because the cases came from documents that exist.

What to look for in the output:

- Zero mismatches.
- The `a.htm` case matching while `a1.htm` is rejected, one character apart.

In [5]:
# =============================================================================
# Section 6.0: A test harness for the address pattern
# =============================================================================

# Each case is (address, should_match, why_this_case_exists).
#
# The third field is not decoration. A test you cannot explain is a test you
# will delete the first time it goes red, and the reason you wrote it is the
# only thing that tells you whether the code broke or the test was wrong.
TEST_CASES = [
    # --- Must match: one per address scheme -------------------------------
    (f"{BASE_URL}/boarddocs/press/monetary/2001/20010103/default.htm",
     True, "1997 to 2005 scheme, monetary folder"),

    (f"{BASE_URL}/boarddocs/press/general/1998/19981015/default.htm",
     True, "Same era but filed under general, which the Fed did for some years"),

    (f"{BASE_URL}/boarddocs/press/monetary/2003/20030128/",
     True, "Same era with default.htm left off, which appears on some pages"),

    (f"{BASE_URL}/newsevents/press/monetary/20080122a.htm",
     True, "2006 to 2010 scheme, the era attempts one and two both missed"),

    (f"{BASE_URL}/newsevents/pressreleases/monetary20240131a.htm",
     True, "2011 onward scheme, monetary glued to the front of the date"),

    # --- Must reject: other documents from the same meeting ---------------
    # These are the dangerous ones. They share the date, they sit in the same
    # folder, and they differ from the statement by one or two characters.
    (f"{BASE_URL}/newsevents/pressreleases/monetary20240131a1.htm",
     False, "Implementation note. One character past the statement address"),

    (f"{BASE_URL}/newsevents/pressreleases/monetary20240131b.htm",
     False, "Longer-run goals and strategy, a separate document"),

    (f"{BASE_URL}/newsevents/pressreleases/monetary20240131c.htm",
     False, "Balance sheet principles, also separate"),

    (f"{BASE_URL}/newsevents/pressreleases/monetary20240131a.pdf",
     False, "PDF mirror of the statement. Same content, wrong format for us"),

    # --- Must reject: other FOMC output -----------------------------------
    (f"{BASE_URL}/monetarypolicy/fomcminutes20240131.htm",
     False, "Minutes, published three weeks later. Not a point-in-time document"),

    (f"{BASE_URL}/monetarypolicy/files/fomcprojtabl20240131.pdf",
     False, "Projection tables"),

    (f"{BASE_URL}/monetarypolicy/files/FOMCpresconf20240131.pdf",
     False, "Press conference transcript"),

    (f"{BASE_URL}/monetarypolicy/fomc20080625.htm",
     False, "Meeting page. Contains the word monetary and a date, matches neither pattern"),

    # --- Must reject: decoys from other release families ------------------
    # Different corner of the same site, similar shape. If a pattern is too
    # loose these are what it pulls in.
    (f"{BASE_URL}/fomc/beigebook/2008/20081015/default.htm",
     False, "Beige Book. Very close to the pre-2006 statement shape"),

    (f"{BASE_URL}/newsevents/pressreleases/bcreg20240131a.htm",
     False, "Banking regulation release. Identical shape, different prefix"),
]


def run_harness(patterns, cases):
    """
    Check a set of patterns against the test cases.

    Returns the failures only. A passing test has nothing to tell you, and
    printing all fifteen results every run trains you to skim the output.
    """
    failures = []
    for address, should_match, reason in cases:
        # Same matching rule the scraper uses: matched by any pattern is a hit.
        matched = any(pattern.search(address) for pattern in patterns)
        if matched != should_match:
            failures.append({
                "address": address.replace(BASE_URL, ""),
                "expected": "match" if should_match else "reject",
                "got": "match" if matched else "reject",
                "reason": reason,
            })
    return failures


CURRENT_PATTERNS = [RE_FILENAME, RE_LEGACY]

print(f"Running {len(TEST_CASES)} cases against the current patterns\n")
failures = run_harness(CURRENT_PATTERNS, TEST_CASES)

if not failures:
    print(f"All {len(TEST_CASES)} cases passed.")
else:
    print(f"{len(failures)} of {len(TEST_CASES)} cases failed:\n")
    print(pd.DataFrame(failures).to_string(index=False))

# Re-run the patterns we already know were wrong. The harness has to fail on
# those, or it is not testing anything. A test suite that passes on broken code
# is worse than no test suite, because it tells you the code is fine.
print("\nSanity check: the harness against attempt one and attempt two")
for label, patterns in [("attempt one", [RE_MODERN_ATTEMPT1, RE_LEGACY]),
                        ("attempt two", [RE_FILENAME_ATTEMPT2, RE_LEGACY])]:
    caught = run_harness(patterns, TEST_CASES)
    print(f"  {label}: {len(caught)} failures")

Running 15 cases against the current patterns

All 15 cases passed.

Sanity check: the harness against attempt one and attempt two
  attempt one: 1 failures
  attempt two: 1 failures


All fifteen pass, and the sanity check does its job: attempt one and attempt two
each fail exactly one case, the 2006 to 2010 address.

That single case is also the measure of how thin the protection is:

- One test out of fifteen stood between the current pattern and the one that lost
  the financial crisis.
- That test exists only because we already found the bug the slow way.
- Had I written this harness before running the scrape, I would have written
  cases for the two schemes I knew about and it would have passed on everything.

We now have 235 addresses and a pattern we can change without fear. Next we
download the documents, which is where the second kind of failure lives.

## 7.0 Fetching and the Encoding Trap

We have 235 addresses. Downloading them is straightforward, and one step in the
middle is not.

A web server sends bytes. Bytes become text only when you apply a character set,
meaning the table saying which byte is which character. The server declares which
one it used, `requests` reads that declaration, and `response.text` hands you a
string. That path works nearly everywhere.

It does not work here. Parts of the Fed's site declare `iso-8859-1` while serving
UTF-8, a mismatch that produces text rather than an error:

- Plain letters and digits are identical in both, so most of the page decodes
  correctly.
- Characters outside that range, meaning en-dashes, curly quotes, and accented
  letters, come out as two or three wrong characters instead of one right one.

The result reads as almost correct, which is the problem. A statement with a
corrupted en-dash in the vote tally is still a statement, still the right length,
and still passes any check you would think to write.

Why it matters for what we are building:

- Word counts shift, because a corrupted character can split or join tokens.
- A dictionary lookup fails on a word that now carries a stray character.
- Comparing two statements picks up differences that are decoding artifacts
  rather than editorial changes.

None of that raises an error. It shows up as a feature slightly wrong in a way
that correlates with document age, which is the worst kind of wrong for a
backtest.

The fix is to stop trusting the declaration and decode from the raw bytes, trying
UTF-8 first and falling back to `latin-1` only when UTF-8 genuinely fails. This
is why Section 4.0 cached bytes rather than text: the correction runs against
files already on disk.

What to look for in the output:

- How many of the 235 pages decode differently under the two approaches.
- Whether the damage is visible to a simple marker check, or only to a character
  by character comparison.

In [6]:
# =============================================================================
# Section 7.0: Comparing the two ways of turning bytes into text
# =============================================================================

# Characters that show up when UTF-8 bytes are read as iso-8859-1. An en-dash
# is three bytes in UTF-8, and reading those bytes one at a time under the
# wrong table surfaces them as â, €, and a stray control character. Searching
# for these is a cheap check you can run against any scraped corpus.
MOJIBAKE_MARKERS = "â|Â|€"


def decode_declared(raw, details):
    """
    The normal approach: use whichever character set the server declared.

    This is what response.text does internally and what almost every scraping
    example you will find does. It is correct whenever the declaration is
    correct.
    """
    return raw.decode(details.get("declared_encoding") or "utf-8", errors="replace")


def decode_strict(raw):
    """
    Decode from the bytes, ignoring what the server claimed.

    UTF-8 first, because it is what the site actually serves. The important
    part is that we do NOT pass errors="replace" here: if the bytes are not
    valid UTF-8 we want the exception, because that is the signal to fall back
    rather than something to paper over.

    latin-1 as the fallback because it never fails. Every byte value maps to
    some character, so this always returns a string. That makes it a safe last
    resort and a terrible first choice.
    """
    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        return raw.decode("latin-1")


# --- Download the statements themselves --------------------------------------
# Section 4.0 fetched the calendar pages. This fetches the 235 documents those
# pages point at, and it is the one place in the notebook where the full
# one second delay applies to every request. Budget four to five minutes.
print(f"Fetching {len(urls_attempt3)} statements. This is the slow cell.\n")

raw_pages = {}
for n, (stamp, url) in enumerate(urls_attempt3.items(), start=1):
    raw, details = fetch_raw(url, session)
    if raw is None:
        print(f"  FETCH FAILED: {stamp.date()} {url}")
        continue
    raw_pages[stamp] = (raw, details)
    if n % 50 == 0:
        print(f"  {n} of {len(urls_attempt3)}")

print(f"\nPages retrieved: {len(raw_pages)}")
print(f"Fetch failures : {len(urls_attempt3) - len(raw_pages)}\n")


# --- Compare the two decodings -----------------------------------------------
# A page where the two approaches produce different text is a page where the
# declaration was wrong. Identical bytes decoded correctly two ways would
# produce identical strings.
affected = []
for stamp, (raw, details) in raw_pages.items():
    wrong = decode_declared(raw, details)
    right = decode_strict(raw)
    if wrong != right:
        affected.append({
            "date": stamp.date(),
            "declared": details.get("declared_encoding"),
            # Whether the damage is visible to the marker check, or whether it
            # is the quieter kind that only shows up in a character comparison.
            "visible_artifacts": bool(re.search(MOJIBAKE_MARKERS, wrong)),
        })

print(f"Pages where the two decodings disagree: {len(affected)} of {len(raw_pages)}")

if affected:
    df_affected = pd.DataFrame(affected)
    print(f"Of those, with visible artifacts     : {df_affected['visible_artifacts'].sum()}")
    print("\nDeclared character set on the affected pages:")
    print(df_affected["declared"].value_counts().to_string())
    print("\nFirst ten affected statements:")
    print(df_affected.head(10).to_string(index=False))
else:
    print("No disagreement. Every declaration on this site is currently correct.")

Fetching 235 statements. This is the slow cell.

  50 of 235
  100 of 235
  150 of 235
  200 of 235

Pages retrieved: 235
Fetch failures : 0

Pages where the two decodings disagree: 174 of 235
Of those, with visible artifacts     : 174

Declared character set on the affected pages:
declared
ISO-8859-1    174

First ten affected statements:
      date   declared  visible_artifacts
2006-01-31 ISO-8859-1               True
2006-03-28 ISO-8859-1               True
2006-05-10 ISO-8859-1               True
2006-06-29 ISO-8859-1               True
2006-08-08 ISO-8859-1               True
2006-09-20 ISO-8859-1               True
2006-10-25 ISO-8859-1               True
2006-12-12 ISO-8859-1               True
2007-01-31 ISO-8859-1               True
2007-03-21 ISO-8859-1               True


### 7.1 Seeing the Damage

174 pages is not a corner case, it is most of the corpus. Before fixing it we
look at what the corruption does to a specific line, because a count tells you
how many and an example tells you what you would have been trading on.

The cell below takes one affected statement, decodes it both ways, and prints
where the two versions first disagree.

What to look for in the output:

- The corrupted line above the correct one, with the character that broke.
- Whether you would have spotted it reading the statement normally.

In [7]:
# =============================================================================
# Section 7.0: What the corruption looks like in a specific statement
# =============================================================================

def difference_positions(wrong_text, right_text):
    """
    Every position where two decodings of the same page disagree.

    Returning all of them rather than the first, because the first difference
    is often a byte order mark at position zero, which is real but not what a
    reader would notice.
    """
    return [i for i, (a, b) in enumerate(zip(wrong_text, right_text)) if a != b]


def show_window(text, pos, width=70):
    """
    Text around a position, with the slice bounds clamped.

    max(0, ...) matters here: a negative start index does not raise, it wraps
    to the end of the string and returns something that looks like text but is
    from entirely the wrong place.
    """
    return text[max(0, pos - width):pos + width].replace("\n", " ")


example_date = pd.Timestamp(df_affected["date"].iloc[0])
raw, details = raw_pages[example_date]

wrong = decode_declared(raw, details)
right = decode_strict(raw)

positions = difference_positions(wrong, right)
print(f"Statement       : {example_date.date()}")
print(f"Server declared : {details.get('declared_encoding')}")
print(f"Characters that differ: {len(positions)} of {len(right):,}")
print(f"Positions       : {positions[:10]}\n")

for pos in positions[:4]:
    print(f"--- position {pos} ---")
    print(f"  declared charset : ...{show_window(wrong, pos)}...")
    print(f"  bytes as UTF-8   : ...{show_window(right, pos)}...")
    print(f"  repr             : {wrong[pos]!r} instead of {right[pos]!r}\n")


# The marker check missed most of this. Compare what each detector finds on
# the same 174 pages: markers tuned to one corruption against a direct
# comparison of the two decodings.
detector_comparison = []
for stamp, (r, d) in raw_pages.items():
    w = decode_declared(r, d)
    s = decode_strict(r)
    detector_comparison.append({
        "date": stamp.date(),
        "marker_hits": len(re.findall(MOJIBAKE_MARKERS, w)),
        "chars_differing": sum(1 for a, b in zip(w, s) if a != b),
        "pct_of_document": round(100 * sum(1 for a, b in zip(w, s) if a != b) / max(len(s), 1), 1),
    })

df_detect = pd.DataFrame(detector_comparison)
df_detect = df_detect[df_detect["chars_differing"] > 0]

print(f"Pages with any corruption: {len(df_detect)}")
print(f"Median share of the document affected: {df_detect['pct_of_document'].median():.1f}%")
print(f"Pages where the marker check found nothing: {(df_detect['marker_hits'] == 0).sum()}")
print("\nSample:")
print(df_detect.head(10).to_string(index=False))

Statement       : 2006-01-31
Server declared : ISO-8859-1
Characters that differ: 53098 of 80,762
Positions       : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

--- position 0 ---
     ...
 ...
  repr             : 'ï' instead of '\ufeff'

--- position 1 ---
...
  ...
  repr             : '»' instead of '<'

--- position 2 ---
 ...
   ...
  repr             : '¿' instead of '!'

--- position 3 ---
  ...
    ...
  repr             : '<' instead of 'd'

Pages with any corruption: 174
Median share of the document affected: 66.5%
Pages where the marker check found nothing: 0

Sample:
      date  marker_hits  chars_differing  pct_of_document
2006-01-31            2            53098             65.7
2006-03-28            2            53408             65.9
2006-05-10            2            55162             66.5
2006-06-29            2            53472             65.9
2006-08-08            2            53049             65.7
2006-09-20            2            53015             65.7
2006-10-25          

174 of 235 pages, and the median page has 66% of its characters wrong.

Position zero shows the mechanism:

| Position | Read as declared | Correct |
| --- | --- | --- |
| 0 | `ï` | `\ufeff` |
| 1 | `»` | `<` |
| 2 | `¿` | `!` |
| 3 | `<` | `d` |

The first three bytes are a byte order mark, which should decode to one invisible
character. Read one byte at a time under the wrong table it becomes three visible
ones, everything after is shifted by two, and the shift never recovers. This is
not a document with a few damaged characters, it is a document wrong from the
first character to the last.

Now the part worth sitting with:

- Characters corrupted per page: about 53,000
- Marker check hits per page: 2

The check ran on all 174 pages and returned a positive result on all 174. It was
not silent and it was not wrong. It looked for `â`, `Â`, and `€`, the characters
an en-dash produces when misread, and found the two of those that happened to be
present. A byte order mark produces different characters entirely, so the
detector had no view of the corruption doing the damage.

That is a distinct failure from the earlier ones:

- **Sections 4.0 and 5.0:** the check was missing, so nothing was reported.
- **Here:** the check was present, ran correctly, and reported a number four
  orders of magnitude too small.

A detector tuned to one kind of corruption will quietly report a low count on a
different kind. The number feels like a measurement of the problem and is
actually a measurement of the detector.

We now decode from the bytes for every page in the corpus.

### 7.2 Decoding the Corpus Correctly

We decode all 235 pages from the bytes, then run both detectors again to confirm
the corruption is gone.

One further step goes in at the same time. The statements contain typographic
punctuation, meaning curly quotes and dashes rather than the plain ASCII
versions. Those decode correctly now and are still a problem for what comes
later:

- A lookup for `don't` will not match `don't`, since the apostrophes are
  different characters.
- Splitting text on punctuation behaves differently for an en-dash than for a
  hyphen.
- The Fed's usage changed over thirty years, so the same word can tokenize one
  way in 1999 and another in 2019.

`unicodedata.normalize` with the `NFKC` form maps most of these to ASCII in one
pass. It is standard library, deterministic, and running it now means every
feature we build later sees consistent tokens.

What to look for in the output:

- Zero pages disagreeing between the two decodings, since we now use one.
- Zero marker hits.
- The normalization count, which tells you how many pages carried typographic
  punctuation.

In [8]:
# =============================================================================
# Section 7.0: Decoding the full corpus and normalizing punctuation
# =============================================================================

def normalize_text(text):
    """
    Map typographic punctuation to ASCII equivalents and drop the byte order
    mark.

    NFKC is the "compatibility composition" normalization form. The part we
    want is that it rewrites characters that are stylistic variants of a
    simpler character: curly quotes become straight quotes, and various
    dashes collapse toward the plain hyphen.

    The BOM is removed separately, because NFKC leaves it alone. It is a
    legitimate character at the start of a file and it is invisible, which is
    exactly why it causes trouble: any pattern anchored with ^ to the start of
    the document will not match past it, and nothing in the output shows you
    why.
    """
    return unicodedata.normalize("NFKC", text).lstrip("\ufeff")


# Decode every page from its bytes and normalize in the same pass. This is the
# text the rest of the notebook works from, so raw_pages is not used again
# except where we deliberately want the undecoded version.
pages = {}
normalized_count = 0

for stamp, (raw, details) in raw_pages.items():
    decoded = decode_strict(raw)
    normalized = normalize_text(decoded)
    if normalized != decoded:
        normalized_count += 1
    pages[stamp] = normalized

print(f"Pages decoded    : {len(pages)}")
print(f"Pages changed by normalization: {normalized_count}\n")

# --- Confirm both detectors now come back clean ------------------------------
# Detector one: the marker search, which we now know understates the scale but
# is still a valid test for the specific characters it looks for.
marker_hits = sum(len(re.findall(MOJIBAKE_MARKERS, text)) for text in pages.values())
pages_with_markers = sum(1 for text in pages.values() if re.search(MOJIBAKE_MARKERS, text))

# Detector two: the comparison that actually measured the problem. Decoding
# the bytes strictly and comparing against the text we are keeping should
# produce no differences at all, since one is derived from the other. This is
# a check that the pipeline does what we think, not a check on the Fed.
disagreements = 0
for stamp, (raw, details) in raw_pages.items():
    if normalize_text(decode_strict(raw)) != pages[stamp]:
        disagreements += 1

print("After the fix:")
print(f"  Marker hits across the corpus : {marker_hits}")
print(f"  Pages containing a marker     : {pages_with_markers} of {len(pages)}")
print(f"  Decoding disagreements        : {disagreements} of {len(pages)}")

# The byte order mark specifically, since that is what did the damage. It is
# legitimate at the very start of a file and meaningless anywhere else, so we
# check both positions separately.
bom_leading = sum(1 for text in pages.values() if text.startswith("\ufeff"))
bom_internal = sum(1 for text in pages.values() if "\ufeff" in text[1:])
print(f"  Pages starting with a BOM     : {bom_leading}")
print(f"  Pages with a BOM further in   : {bom_internal}")

Pages decoded    : 235
Pages changed by normalization: 175

After the fix:
  Marker hits across the corpus : 0
  Pages containing a marker     : 0 of 235
  Decoding disagreements        : 0 of 235
  Pages starting with a BOM     : 0
  Pages with a BOM further in   : 0


Both detectors come back clean, and the fix ran against files already on disk
rather than against the Fed's servers, which is the payoff for caching bytes back
in Section 4.0.

The byte order mark is the small version of the same lesson. It survived the
first fix because it is a legitimate character, it is invisible in any output you
would print, and it would have broken any pattern anchored to the start of the
document without ever explaining itself.

We have 235 correctly decoded pages. Next we pull the statement text out of them,
which is where the length checks come in.

## 8.0 Extracting the Statement Text

We have 235 correctly decoded pages, and they fall into two groups needing
different handling:

- **Modern and mid-era pages** wrap the release in a container element with the
  id `article`. Take the paragraphs inside it and you have the statement.
- **Pre-2006 pages** predate that convention. Nothing separates the statement
  from the navigation, the site's own index of releases, and the footer, and all
  of it is formatted as paragraphs.

The older pages therefore need guards, meaning rules that recognize text which is
not part of a statement:

- **Anchor ratio.** A paragraph that is mostly hyperlink text is a link list. We
  measure the share of a paragraph's characters sitting inside `<a>` tags and
  drop anything above 60%.
- **Site furniture markers.** The release index that follows the statement opens
  with recognizable headings. We truncate at the earliest one found, since
  everything after the first is also not the statement.
- **Minimum length.** Paragraphs under 40 characters are labels and captions.

One thing we deliberately do not do is strip `<table>` elements. Removing tables
is the standard cleanup for old HTML, and here it would delete the document,
because pre-2006 pages use tables for layout and the statement can sit inside
one.

Two wrong turns are worth recording, since both produced output that looked
reasonable:

- **Every paragraph on every page, no container.** Legacy median 1,158 words
  against statements that ran nearer 200. It also inflated the modern median to
  686, which is the more instructive part: modern pages have a container and the
  extractor was not consulting it.
- **Selecting the container by Bootstrap grid classes.**
  `col-xs-12 col-sm-8 col-md-8` matched a 15-word fragment and collapsed the
  modern median to 10 words. Grid classes describe how wide an element renders,
  several elements carry them, and `select_one` returns whichever comes first. An
  id describes content and is meant to be unique, which is why `div#article` is
  the selector here.

The detector for both was document length. Statements from the late 1990s ran
roughly 150 to 250 words and modern ones nearer 450, numbers that come from
having read them rather than from anything the parser produces.

What to look for in the output:

- Median words per era against those ranges.
- The legacy maximum, since one long page surviving is a different problem from
  the whole era being long.
- Bodies under 50 words, which must be zero. An over-aggressive guard will not
  show up in a median.

In [9]:
# =============================================================================
# Section 8.0: Extracting the statement text
# =============================================================================

# Which of the three URL schemes a statement came from. We group by this rather
# than by year because the page structure changed with the site reorganizations,
# not on any calendar boundary, so the era is what predicts how a page parses.
def url_era(url):
    if "/boarddocs/" in url:
        return "1 legacy"
    if "/newsevents/press/monetary/" in url:
        return "2 mid"
    return "3 modern"


# Selected by id rather than by layout class. See the note in the section
# intro: grid classes such as col-sm-8 appear on several elements per page and
# change with any restyle of the site.
ARTICLE_SELECTORS = ["div#article"]

# Headings the Fed's older pages place after the statement, at the start of the
# site's own list of other releases. Cutting at the earliest one is more
# reliable than removing each individually, since whatever follows the first
# marker is also not the statement.
FURNITURE_MARKERS = [
    "Home | News and events",
    "Home | News",
    "Accessibility",
    "Last update:",
    "Return to top",
    "Board of Governors of the Federal Reserve System",
    "Federal Reserve Board of Governors",
    "Other Federal Reserve Board Sites",
    "Related Press Releases",
    "Recent Postings",
    "Skip to main content",
]


def find_article(soup):
    """
    Return the container holding the release, or None if the page has none.

    Returning None rather than falling back inside this function is
    deliberate. The caller needs to know whether a container was found,
    because that is the difference between an extraction we trust and one we
    have to guard.
    """
    for selector in ARTICLE_SELECTORS:
        node = soup.select_one(selector)
        if node is not None:
            return node
    return None


def anchor_ratio(paragraph_tag):
    """
    Share of a paragraph's characters that sit inside a hyperlink.

    A statement links almost nothing, so its paragraphs score near zero. A
    navigation strip scores near one. The threshold sits at 0.60 to leave room
    for a statement that happens to contain one link.
    """
    total = len(paragraph_tag.get_text(" ", strip=True))
    if total == 0:
        return 1.0
    linked = sum(len(a.get_text(" ", strip=True)) for a in paragraph_tag.find_all("a"))
    return linked / total


def truncate_at_furniture(text):
    """
    Cut the text at the earliest site furniture marker.

    Earliest rather than first-in-list. Checking the markers in the order we
    happened to write them would cut at whichever we listed first, which is
    not necessarily the one appearing first in the document.
    """
    positions = [text.find(marker) for marker in FURNITURE_MARKERS]
    positions = [p for p in positions if p != -1]
    return text[:min(positions)].strip() if positions else text


def extract_body(html):
    """
    Extract the statement, using the container when the page has one and
    guarded paragraph extraction when it does not.

    Returns (text, used_container) so we can count how many pages took each
    path. A fix that silently applies to fewer pages than you think is the
    kind of thing this notebook keeps running into.

    Note what is absent: we do not remove <table> elements, because pre-2006
    pages use tables for layout and the statement can sit inside one.
    """
    soup = BeautifulSoup(html, HTML_PARSER)
    node = find_article(soup)
    used_container = node is not None

    if used_container:
        # The container already excludes navigation and footer, so the guards
        # would only risk dropping legitimate text.
        paragraphs = [p.get_text(" ", strip=True) for p in node.find_all("p")]
        return "\n\n".join(p for p in paragraphs if p), True

    kept = []
    for p in soup.find_all("p"):
        # get_text(" ", strip=True) joins the pieces inside a paragraph with a
        # space. Without the separator, text broken across inline tags runs
        # together into a single made-up word.
        text = p.get_text(" ", strip=True)
        if len(text) < MIN_PARAGRAPH_CHARS:
            continue
        if anchor_ratio(p) > MAX_ANCHOR_TEXT_RATIO:
            continue
        kept.append(text)

    return truncate_at_furniture("\n\n".join(kept)), False


# Build the corpus table. Everything from here on is a DataFrame rather than a
# dictionary, since we now have several fields per statement and want to
# measure across them.
records = []
for stamp, html in pages.items():
    url = urls_attempt3[stamp]
    body, used_container = extract_body(html)
    records.append({
        "date": stamp,
        "url": url,
        "era": url_era(url),
        "body": body,
        "used_container": used_container,
        # Word count on whitespace, which is crude and consistent. We are
        # comparing documents against each other, so the exact tokenization
        # matters less than using the same rule for all of them.
        "n_words": len(body.split()),
    })

df_stmt = pd.DataFrame(records).sort_values("date").reset_index(drop=True)

print(f"Statements extracted: {len(df_stmt)}\n")

print("Pages that found an article container, by era:")
print(df_stmt.groupby("era")["used_container"].agg(["sum", "count"]).to_string())

print("\nWord count by era:")
print(df_stmt.groupby("era")["n_words"]
      .agg(["count", "median", "min", "max"]).round(0).to_string())

print("\n  Legacy statements ran roughly 150 to 250 words, modern nearer 450.")

# A guard that removes too much will not show up in a median.
too_short = df_stmt[df_stmt["n_words"] < 50]
print(f"\nBodies under 50 words: {len(too_short)}")
if len(too_short):
    print(too_short[["date", "era", "n_words"]].to_string(index=False))


Statements extracted: 235

Pages that found an article container, by era:
          sum  count
era                 
1 legacy    0     61
2 mid      43     43
3 modern  131    131

Word count by era:
          count  median  min   max
era                               
1 legacy     61   206.0  107   357
2 mid        43   295.0   85  1394
3 modern    131   462.0  145   910

  Legacy statements ran roughly 150 to 250 words, modern nearer 450.

Bodies under 50 words: 0


Every era lands where it should:

| Era | Count | Median | Min | Max |
| --- | --- | --- | --- | --- |
| 1 legacy | 61 | 206 | 107 | 357 |
| 2 mid | 43 | 295 | 85 | 1,394 |
| 3 modern | 131 | 462 | 145 | 910 |

Reading the table:

- **Legacy at 206 words** is inside the range those statements actually ran, down
  from 1,158 under the first version of this code.
- **The container column** confirms where each path applied: zero of 61 legacy
  pages have a container and all 174 mid and modern pages do, so the guards ran
  only where needed.
- **The legacy maximum of 357** means no single page is still dragging in the
  release index. Had the median come down while the maximum stayed near 2,000,
  the guards would be working on most pages and failing on a few, which is harder
  to spot and worse to leave in.
- **The mid maximum of 1,394** is not extraction. That is the March 2008
  coordinated central bank action, genuinely that long, and Section 10.0 flags
  rather than removes it.
- **Modern shows 131 against the 128** the finished corpus will hold. Three of
  those are not policy statements and they stay in until Section 10.0, so you can
  see what they do to these numbers.

Extraction is settled. What the length check could never tell us is whether the
text inside those bodies is correctly separated into its parts, which is the next
section.

## 9.0 Splitting the Voting Record

Each statement ends with the Committee's vote, naming who voted for the policy
action and, when there is disagreement, who voted against it and why.

That block needs separating from the statement text, for a reason specific to
what we are building:

- Any measure of the statement's language is affected by a list of names.
- The list changes with committee membership rather than with policy.
- A statement naming four dissenters is not more or less hawkish than one naming
  none, but a word-based measure will score it differently.

So we split the body into statement and voting record, and store them separately.

This is the section where I got something wrong that mattered. The first three
failures produced bad data, which is recoverable once you find it. This one
produced a conclusion, which I wrote down and believed, and the data behind it
was fine. I want you to watch that happen rather than take my word for it,
because the distance between a parser bug and a research finding is shorter than
it feels.

What to look for in the output:

- How many statements the direct approach finds a voting record in.
- Which statements it does not, and what year they fall in.

In [10]:
# =============================================================================
# Section 9.0: Splitting the voting record, first attempt
# =============================================================================

# The phrase the Fed uses to introduce the vote. Everything from here to the
# end of the body is the voting record rather than the statement.
VOTE_MARKER = "Voting for the monetary policy action"


def split_votes_literal(body):
    """
    Split the body at the vote marker.

    Returns (statement, votes). An empty votes string means the marker was not
    found, and the whole body is treated as statement text.
    """
    position = body.find(VOTE_MARKER)
    if position == -1:
        return body, ""
    return body[:position].strip(), body[position:].strip()


statements, votes = zip(*(split_votes_literal(b) for b in df_stmt["body"]))
df_stmt["statement"] = statements
df_stmt["votes"] = votes

found = (df_stmt["votes"].str.len() > 0).sum()
print(f"Voting record found: {found} of {len(df_stmt)}\n")

missing = df_stmt[df_stmt["votes"].str.len() == 0]
print("Statements with no voting record, by year:")
print(missing.groupby(missing["date"].dt.year).size().to_string())

Voting record found: 57 of 235

Statements with no voting record, by year:
date
1997     1
1998     3
1999     6
2000     8
2001    11
2002     8
2003     8
2004     8
2005     8
2006     8
2007     9
2008     9
2009     8
2010     9
2011     8
2012     8
2013     8
2014     8
2015     8
2016     8
2017     8
2018     8
2019     4
2020     3
2025     1
2026     2


57 of 235, and the shape of the failure is not what a merely narrow marker would
produce:

- **1997 through 2018:** every statement missing, eight or nine per year.
- **2019 through 2026:** most found, a handful missing.

A marker matching only the last few years means the phrasing changed at some
point and I wrote down the version I had most recently read. A straightforward
mistake, and the year table makes it visible immediately.

Now look at what the same output would have shown had the marker been closer to
correct:

- Suppose it matched everything up to 2025 and missed only the last few
  statements.
- The table would show two or three missing rows at the bottom.
- Those rows would carry a story, since the Committee changed hands in May 2026.

That is the situation I was actually in, and it is worth reconstructing rather
than describing, because the output looks reasonable and the conclusion follows
from it.

### 9.1 Finding the Phrasing the Fed Actually Uses

Rather than guess at a second marker, we find every phrasing in the corpus. The
word "Voting" appears in the vote block and almost nowhere else, so we can pull
the sentence around each occurrence and see the variants directly.

What to look for in the output:

- How many distinct phrasings exist across thirty years.
- Which are near-identical variations, and which are structurally different.

In [11]:
# =============================================================================
# Section 9.0: What phrasings does the corpus actually contain?
# =============================================================================

# Match "Voting" and capture what follows, up to the first name. The
# {0,60} bound keeps the capture short enough to group meaningfully: without
# it, every match is unique because every roster is unique.
RE_VOTE_CONTEXT = re.compile(r"(Voting[^.]{0,60}?)\s+(?:were|was)\b", re.IGNORECASE)
# Same pattern without the capture group. str.contains ignores groups and
# warns about them, so the boolean test below uses this version while
# finditer above keeps the grouped one.
RE_VOTE_ANY = re.compile(r"Voting[^.]{0,60}?\s+(?:were|was)\b", re.IGNORECASE)

variants = []
for body in df_stmt["body"]:
    for match in RE_VOTE_CONTEXT.finditer(body):
        variants.append(match.group(1).strip())

print(f"Occurrences of a voting phrase: {len(variants)}\n")
print("Distinct phrasings, most common first:")
print(pd.Series(variants).value_counts().to_string())

# Statements where the word appears at all but the phrasing above did not
# match. These are the structurally different cases rather than wording
# variations.
has_word = df_stmt["body"].str.contains(r"\bVoting\b", case=False, regex=True)
matched = df_stmt["body"].str.contains(RE_VOTE_ANY, regex=True)
odd = df_stmt[has_word & ~matched]
print(f"\nStatements containing 'Voting' but not matching the pattern: {len(odd)}")
for _, r in odd.iterrows():
    idx = r["body"].lower().find("voting")
    print(f"  {r['date'].date()}: ...{r['body'][idx:idx + 90]}...")

# And the statements with no occurrence of the word at all.
none_at_all = df_stmt[~has_word]
print(f"\nStatements with no occurrence of 'Voting': {len(none_at_all)}")
print(none_at_all.groupby(none_at_all["date"].dt.year).size().to_string())

# Which recent statements have no occurrence of the word? Named explicitly,
# because the next subsection is built on these specific documents.
print("\nStatements from 2019 onward with no occurrence of 'Voting':")
recent_none = none_at_all[none_at_all["date"].dt.year >= 2019]
print(recent_none[["date", "era", "n_words"]].to_string(index=False))

Occurrences of a voting phrase: 274

Distinct phrasings, most common first:
Voting for the FOMC monetary policy action             137
Voting for the monetary policy action                   57
Voting against the action                               41
Voting against                                          14
Voting against this action                              13
Voting against the policy                                4
Voting against the policy action                         4
Voting for the FOMC monetary policy actions              1
Voting (by notation) for the monetary policy action      1
voting                                                   1
Voting against the monetary policy action                1

Statements containing 'Voting' but not matching the pattern: 0

Statements with no occurrence of 'Voting': 39
date
1997     1
1998     3
1999     6
2000     8
2001    11
2002     1
2007     1
2008     1
2009     1
2010     1
2019     1
2020     2
2025     1
2026     1

Stat

Eleven phrasings across 274 occurrences:

| Phrasing | Count |
| --- | --- |
| Voting for the FOMC monetary policy action | 137 |
| Voting for the monetary policy action | 57 |
| Voting against the action | 41 |
| Voting against | 14 |
| Voting against this action | 13 |
| Voting against the policy | 4 |
| Voting against the policy action | 4 |
| Voting for the FOMC monetary policy actions | 1 |
| Voting (by notation) for the monetary policy action | 1 |
| voting | 1 |
| Voting against the monetary policy action | 1 |

Two structural forms, "Voting for" and "Voting against", with nine variations in
the words between. They are not a drift over time toward a standard, they recur
throughout, and three appear exactly once in thirty years.

That count of one is what makes enumeration the wrong approach:

- `Voting (by notation)` occurs in one statement, 2020-03-23, an emergency
  intermeeting action taken by phone.
- Had that statement not been in the corpus I would never have known the variant
  existed.
- A marker list built from the other 234 documents would have been complete right
  up until the next unusual meeting.

The fix is to stop listing variants and match the structure they share.

The 39 statements with no occurrence of the word are a separate question, and the
years matter:

- **1997 through 2001, 29 statements:** the era when statements were shorter and
  did not always record the vote. A real property of those documents.
- **Ten scattered across 2002 to 2026, including 2026-06-17:** not a real
  property, and the subject of the next subsection.

### 9.2 The Finding That Was Not Real

Here is the position I was in, reconstructed from the output above:

- The vote block splits correctly on 195 or so statements.
- A small number do not split, most from the 1990s where the vote was not always
  recorded.
- At the recent end there are a few, and one is 2026-06-17, the first statement
  issued under a new Chair.

A statement from a new Chair with no voting record, when every statement for the
previous twenty five years had one. There is an obvious reading and it is not a
careless one: the Committee changed how it communicates when its leadership
changed. That is a structural break, it has a date attached, and for a project
measuring how statement language evolves it is exactly the kind of thing you want
to know about.

I wrote it down. Then I built on it, because a regime marker is useful and the
next step was to see whether other features broke at the same date.

The cell below runs the check that would have caught it, and it is not a
sophisticated check. It reads the statement.

What to look for in the output:

- Whether the June statement records its vote somewhere other than where the
  parser looked.
- What the following statement does.

In [12]:
# =============================================================================
# Section 9.0: Checking the conclusion against the documents
# =============================================================================

# The two statements issued under the new Chair. If the format changed, both
# should look the same as each other and different from what came before.
for target in ["2026-06-17", "2026-07-29"]:
    row = df_stmt[df_stmt["date"] == pd.Timestamp(target)].iloc[0]
    body = row["body"]

    print("=" * 72)
    print(f"{target}  ({row['n_words']} words)")
    print("=" * 72)

    # The opening, where a tally would appear if there is one.
    print("Opening:")
    print(f"  {body[:260]}\n")

    # The closing, where the roster appears in the standard format.
    print("Closing:")
    print(f"  {body[-260:]}\n")

    # And the direct question: does the word appear anywhere at all?
    print(f"Contains 'Voting': {bool(re.search(r'Voting', body))}")
    print(f"Contains 'vote'  : {bool(re.search(r'vote', body, re.IGNORECASE))}\n")

2026-06-17  (157 words)
Opening:
  June 17, 2026

For release at 2:00 p.m. EDT Share

The Federal Open Market Committee approved the following statement for release by a 12 – 0 vote:

The Committee decided to maintain the target range for the federal funds rate at 3-1/2 to 3-3/4 percent, in sup

Closing:
  art reflecting supply shocks that have driven price increases in certain sectors, including energy. The Committee will deliver price stability.

For media inquiries, please email [email protected] or call 202-452-2955.

Implementation Note issued June 17, 2026

Contains 'Voting': False
Contains 'vote'  : True

2026-07-29  (193 words)
Opening:
  July 29, 2026

For release at 2:00 p.m. EDT Share

The Federal Open Market Committee approved the following statement for release by a 9 – 3 vote:

The Committee decided to maintain the target range for the federal funds rate at 3-1/2 to 3-3/4 percent, in supp

Closing:
   Neel Kashkari, and Lorie K. Logan, who preferred to raise the target ra

The vote is in the June statement. It sits in the opening line:

`The Federal Open Market Committee approved the following statement for release
by a 12 – 0 vote:`

Twelve to zero. A unanimous decision, recorded as a tally rather than a roster,
which is why no names follow it. The parser found nothing because it was
searching for the word "Voting" at the end of the document, and the vote was
expressed as "vote" at the beginning.

July then settles the question. It carries the same opening tally, `by a 9 – 3
vote`, and it also names the three dissenters at the bottom in the standard
format. So the format did change under the new Chair, and it changed by adding a
line, not by removing one. Names appear whenever there is disagreement, exactly
as they always have.

The conclusion I drew was that the Committee had stopped naming voters. What
actually happened is that it started publishing the tally as well, and June was
unanimous.

Two things about this failure are worth separating from the three that came
before it.

**The data was fine.** The corpus was complete, correctly decoded, and correctly
extracted. Every character of that vote was sitting in the body where I could
read it. Nothing was missing and nothing needed re-fetching. What failed was a
`str.find` returning -1, and the -1 was interpreted as a fact about the Federal
Reserve rather than a fact about my search string.

**The output was not wrong.** "No voting record found in the June 2026
statement" is a true statement about what the parser did. Every number on the
screen was accurate. The error happened in the gap between what the code
measured and what I took it to mean, and no amount of testing the code would
have caught it, because the code was doing what it was told.

That gap is where research findings come from, and it does not have a unit test.
The thing that caught it was reading the document, and the reason I did not read
it sooner is that the parser had worked on 195 others.

The habit that follows, and it is the one I would take from this entire notebook
if you take only one: before a null result becomes a finding, open the source
document and look at it. Not the extracted field, not the summary table, the
document. A missing value is a claim about the world and it costs you two minutes
to check.

### 9.3 Matching the Pattern Rather Than the Phrasings

With the phrasings enumerated and the false finding understood, the fix follows
from what the eleven variants have in common: the word "Voting", optionally
something in parentheses, then "for" or "against".

One pattern covers all eleven and covers variants not yet written, which is the
point. The pattern describes the structure the Fed uses rather than the specific
sentences it has used so far.

Two details in the implementation:

- **Cut at the earliest match, not the first one tested.** A statement with
  dissenters contains both a "Voting for" line and a "Voting against" line, and
  the record starts at whichever comes first in the document.
- **Capture the opening tally separately.** It is a different thing from a roster
  and only recent statements have it.

What to look for in the output:

- Statements with a voting record, against the 39 that genuinely contain no
  occurrence of the word.
- Whether any statement still has a vote phrase left in its body, which must be
  zero.

In [13]:
# =============================================================================
# Section 9.0: Matching the structure the phrasings share
# =============================================================================

# What all eleven phrasings have in common:
#
#   Voting          the word itself, capitalized or not
#   \s*             optional whitespace
#   (?:\([^)]{0,40}\)\s*)?   an optional parenthetical, as in "(by notation)".
#                            [^)]{0,40} means up to 40 characters that are not
#                            a closing bracket, which keeps the match inside
#                            one set of parentheses rather than running to the
#                            next one further down the document.
#   (?:for|against) either direction
#   \b              a word boundary, so "form" and "againstor" do not match
#
# The (?: ) groups are non-capturing. We want the position of the match, not
# its parts, and a capture group here would only produce the pandas warning
# we removed earlier.
RE_VOTE = re.compile(
    r"Voting\s*(?:\([^)]{0,40}\)\s*)?(?:for|against)\b",
    re.IGNORECASE,
)

# The opening tally, which recent statements carry and older ones do not.
# Written to accept both the en-dash the Fed uses and a plain hyphen, since
# the normalization in Section 7.0 does not convert en-dashes and a future
# statement might use either.
RE_TALLY = re.compile(r"by\s+a\s+(\d{1,2})\s*[-\u2013\u2014]\s*(\d{1,2})\s+vote", re.IGNORECASE)


def split_votes(body):
    """
    Separate the voting record from the statement text.

    Returns (statement, votes, tally_for, tally_against).

    The record is cut at the earliest vote phrase in the document rather than
    at the first one we test for. A statement with dissenters contains both a
    "Voting for" line and a "Voting against" line, and testing them in the
    order we wrote them would cut at whichever we listed first.
    """
    # Opening tally, extracted before anything is removed.
    tally = RE_TALLY.search(body)
    tally_for = int(tally.group(1)) if tally else None
    tally_against = int(tally.group(2)) if tally else None

    # Earliest vote phrase. finditer walks the document in order, so the
    # first result is the earliest position by construction.
    match = next(RE_VOTE.finditer(body), None)
    if match is None:
        return body.strip(), "", tally_for, tally_against

    return (body[:match.start()].strip(),
            body[match.start():].strip(),
            tally_for,
            tally_against)


results = [split_votes(b) for b in df_stmt["body"]]
df_stmt["statement"] = [r[0] for r in results]
df_stmt["votes"] = [r[1] for r in results]
df_stmt["tally_for"] = [r[2] for r in results]
df_stmt["tally_against"] = [r[3] for r in results]

found = (df_stmt["votes"].str.len() > 0).sum()
print(f"Voting record found : {found} of {len(df_stmt)}")
print(f"Opening tally found : {df_stmt['tally_for'].notna().sum()} of {len(df_stmt)}\n")

# The check that has to be zero: no vote phrase may remain in the statement
# text. If one does, the split cut in the wrong place and the roster is still
# contaminating the language features.
leftover = df_stmt[df_stmt["statement"].str.contains(RE_VOTE, regex=True)]
print(f"Statements with a vote phrase still in the body: {len(leftover)}")
if len(leftover):
    print(leftover[["date", "era"]].to_string(index=False))

# Statements with no record, which should be the 39 that contain no
# occurrence of the word at all.
no_record = df_stmt[df_stmt["votes"].str.len() == 0]
print(f"\nStatements with no voting record: {len(no_record)}")
print(no_record.groupby(no_record["date"].dt.year).size().to_string())

# Where a tally exists, does it agree with the roster? Not a check we can run
# on every statement, but where both are present they should be consistent.
both = df_stmt[(df_stmt["tally_for"].notna()) & (df_stmt["votes"].str.len() > 0)]
print(f"\nStatements carrying both a tally and a roster: {len(both)}")
if len(both):
    print(both[["date", "tally_for", "tally_against"]].to_string(index=False))

Voting record found : 196 of 235
Opening tally found : 2 of 235

Statements with a vote phrase still in the body: 0

Statements with no voting record: 39
date
1997     1
1998     3
1999     6
2000     8
2001    11
2002     1
2007     1
2008     1
2009     1
2010     1
2019     1
2020     2
2025     1
2026     1

Statements carrying both a tally and a roster: 1
      date  tally_for  tally_against
2026-07-29        9.0            3.0


The pattern covers all eleven phrasings:

| Check | Result |
| --- | --- |
| Voting record found | 196 of 235 |
| Vote phrase left in statement text | 0 |
| Statements with no record | 39 |

The 39 with no record are exactly the 39 containing no occurrence of the word,
which is the check that the split is complete rather than merely improved. A
number that went up is not evidence on its own, and a number that went up to the
ceiling set by an independent count is.

The tally column is the more interesting result:

- Two statements out of 235 carry an opening vote tally, both from June and July
  2026.
- That is a real format change, dated to the leadership transition, and it is the
  finding I thought I had at the start of this section.
- July 2026 is the only statement carrying both a tally and a roster, `9 - 3`
  with three dissenters named, and the two agree.

One consistency check on one document is not much, and it is available for free
and confirms the two fields mean what we think they mean.

Two statements is too small a sample to build anything on. It is worth recording
in the corpus anyway, because a format that is two statements old today will not
be in two years, and a corpus that captured the field from the start is worth
more than one that has to be rebuilt.

### 9.4 Why Not Have a Language Model Do the Extraction?

A fair question after four sections of pattern matching, particularly in a
chapter about language models. Sending each statement to a model with the
instruction "return the voting record" would have produced working output in an
afternoon.

Two versions of the question, with different answers.

**Could a model have written this code?** Yes, and Section 6.0 said so. Every
pattern here is within reach of a model given a clear description, and the ones I
wrote by hand took longer than asking would have. What it could not do was tell
me the Fed used three URL schemes, or that a 2020 statement says "Voting (by
notation)", because those are facts about specific documents rather than facts
about regex.

**Could a model have done the extraction itself?** Yes, and it would have been
the wrong choice here. Four reasons, in order of weight:

- **Verification.** When the pattern missed, the year table named the statements
  and I read them. When a model quietly summarizes instead of extracting, on
  document 147 of 235, nothing tells you. Catching it needs a deterministic
  checker, which is the code you were trying not to write.
- **Determinism.** Two runs of a regex over the same bytes give the same corpus.
  Two runs through a model may not. Chapter 8 spent its length on backtests being
  reproducible, and a dataset that shifts between runs cannot support one.
- **The task has an exact answer.** Eleven phrasings reduce to two structural
  forms. That is what pattern matching is for, and a correct pattern is correct
  on every future statement without being re-run.
- **Cost.** 235 documents through a model is minutes and a few dollars each time.
  `str.find` is free, and you will re-run this pipeline dozens of times.

Where the balance shifts, and it does shift: a corpus of twenty documents used
once, I would hand to a model and check by hand. The arithmetic changes with
scale and reuse, not with the sophistication of the tool.

Where a model is the only option: extraction with no fixed form. Which sectors of
the economy a statement singles out. Whether its risk language is balanced or
tilted. There is no pattern for those, they are judgments about meaning, and that
is what the rest of this chapter sets out to test.

The division is not old tools against new ones. Structured text gets a parser,
unstructured meaning gets a model, and most real pipelines need both.

## 10.0 Removing Documents That Are Not Policy Statements

The corpus holds 235 documents and not all are policy statements.

The address pattern selects files ending in `a.htm`, the Fed's slot for the main
release from a meeting. Occasionally that slot holds something else: a framework
revision, an announcement about operating procedures, a review of policy
strategy. Those are FOMC releases, they sit at the right address, and they are
not statements about the current stance of policy.

We already have one detector for them without having built it:

- Section 9.0 listed five statements from 2019 onward with no occurrence of the
  word "Voting".
- Three of them were this kind of document.
- That was not what the check was for, and two independent detectors agreeing on
  the same three documents is better evidence than either alone.

A second category needs flagging rather than removing. Some statements announce
coordinated action with other central banks, and those run longer and read
differently because they describe an international agreement. They are genuine
policy communications and they belong in the corpus, so we mark them and let the
analysis in 16.2 decide.

What to look for in the output:

- The three documents identified for removal, and whether their titles support
  it.
- The flagged joint actions, and whether any is a regular meeting caught by
  mistake.

In [14]:
# =============================================================================
# Section 10.0: Identifying documents that are not policy statements
# =============================================================================

# Phrases identifying FOMC releases that are not statements about the current
# policy stance. Restricted to title language: the earlier version included
# "Balance Sheet Normalization" and "Policy Normalization Principles", which
# caught three 2017 statements that discuss the topic rather than documents
# about it. A marker that names a subject will match any statement covering
# that subject, so the markers have to name a document type instead.
NON_POLICY_MARKERS = [
    "Statement on Longer-Run Goals",
    "Longer-Run Goals and Monetary Policy Strategy",
    "Addendum to the Policy Normalization Principles",
    "review of its monetary policy framework",
    "Statement Regarding Monetary Policy Implementation",
    "announced the unanimous",
]

# Language marking a coordinated action with other central banks. These stay
# in the corpus and are flagged rather than dropped.
JOINT_ACTION_MARKERS = [
    "central banks",
    "Bank of Canada",
    "Bank of England",
    "European Central Bank",
    "Swiss National Bank",
    "Bank of Japan",
    "swap line",
    "swap arrangements",
]


def contains_any(text, markers):
    """True when any marker appears in the text. Case sensitive by design:
    the markers are proper nouns and document titles."""
    return any(marker in text for marker in markers)

OPENING_CHARS = 600

# Search the opening only, for the same reason the joint-action check does.
# These are titles, and a title is at the top.
df_stmt["is_non_policy"] = df_stmt["body"].apply(
    lambda b: contains_any(b[:OPENING_CHARS], NON_POLICY_MARKERS)
)

# The joint-action check runs on the opening of the statement only. A
# coordinated action says so at the top. A regular statement that happens to
# mention another central bank further down is not one, and searching the
# whole document produced a false positive in the earlier research.
df_stmt["is_joint_action"] = df_stmt["body"].apply(
    lambda b: contains_any(b[:OPENING_CHARS], JOINT_ACTION_MARKERS)
)

non_policy = df_stmt[df_stmt["is_non_policy"]]
print(f"Documents identified as non-policy: {len(non_policy)}\n")
for _, r in non_policy.iterrows():
    # First 120 characters, which on these pages is the date line and title.
    print(f"  {r['date'].date()} [{r['n_words']} words]")
    print(f"    {r['body'][:120]}\n")

joint = df_stmt[df_stmt["is_joint_action"]]
print(f"Statements flagged as joint central bank actions: {len(joint)}")
print(joint[["date", "era", "n_words"]].to_string(index=False))

Documents identified as non-policy: 3

  2019-10-11 [608 words]
    October 11, 2019

For release at 11:00 a.m. EDT Share

Consistent with its January 2019 Statement Regarding Monetary Pol

  2020-08-27 [426 words]
    August 27, 2020

For release at 9:10 a.m. EDT Share

Following an extensive review that included numerous public events 

  2025-08-22 [206 words]
    August 22, 2025

For release at 10:00 a.m. EDT Share

The Federal Open Market Committee (FOMC) on Friday announced the u

Statements flagged as joint central bank actions: 4
      date      era  n_words
2008-03-11    2 mid     1394
2008-10-08    2 mid      865
2010-05-09    2 mid      286
2020-03-31 3 modern      259


Three documents come out:

| Date | Words | What it is |
| --- | --- | --- |
| 2019-10-11 | 608 | Announcement on monetary policy implementation |
| 2020-08-27 | 426 | Outcome of the framework review |
| 2025-08-22 | 206 | Procedural announcement |

These are the same three the Section 9.0 check found independently, which is the
only confirmation available. There is no external count of non-policy documents
to compare against, so two detectors built for different purposes agreeing is
what stands in for one.

The first version of this check returned six:

- The three extra were policy statements from 2017.
- My marker list included `Balance Sheet Normalization`, a subject the Committee
  was actively discussing that year, so every statement mentioning it matched.
- The correction was to match on document type rather than subject, and to search
  only the opening, since a title is at the top.

The joint-action flag shows the same effect from the other direction:

- Searching the whole document flagged five, including 2010-01-27, a regular
  meeting that mentions swap arrangements in passing.
- Restricted to the opening it flags four, each announcing coordinated action in
  its first sentence.
- The March 2008 entry at 1,394 words is the longest document in the corpus and
  is genuinely that long.

These four stay in. They are real policy communications and dropping them would
remove the Fed's response to the two most severe episodes in the sample, so the
flag lets 16.2 treat them as a category rather than as outliers of unknown
origin.

One caution on the correction itself. The revised marker list is more literal
than the pattern-based approach Section 9.0 arrived at, and `announced the
unanimous` is close to matching an exact phrase. It works on this corpus and will
not generalize to a document type the Fed has not published yet, which is why
Section 11.0 counts the documents rather than trusting the list.

### 10.1 Applying the Removals

We drop the three and keep the flag on the four.

What to look for in the output:

- 232 statements remaining, and the date range.
- Statements per year, which is the same cadence check from Section 4.0 run
  against the finished corpus rather than against a list of addresses.

In [15]:
# =============================================================================
# Section 10.0: Applying the removals
# =============================================================================

# Keep a copy of what is being removed. A dropped row that turns out to have
# been legitimate is easier to recover from a variable than from a re-run.
df_dropped = df_stmt[df_stmt["is_non_policy"]].copy()
df_corpus = df_stmt[~df_stmt["is_non_policy"]].copy().reset_index(drop=True)

print(f"Before removal : {len(df_stmt)}")
print(f"Removed        : {len(df_dropped)}")
print(f"Corpus         : {len(df_corpus)}\n")

print(f"Date range     : {df_corpus['date'].min().date()} to "
      f"{df_corpus['date'].max().date()}")
print(f"Joint actions flagged: {df_corpus['is_joint_action'].sum()}\n")

# The Section 4.0 check, re-run on the finished corpus. The addresses passed it
# earlier, and the question now is whether removing documents has taken the
# count below the cadence in any year.
year_counts = (
    df_corpus["date"].dt.year.value_counts()
    .reindex(range(CORPUS_START.year, CORPUS_END.year + 1), fill_value=0)
    .sort_index()
)

print("Statements per year")
for year, count in year_counts.items():
    # Mark years below the scheduled cadence so they stand out without
    # needing to be read against a number held in your head.
    flag = "  <-- below eight" if (2000 <= year <= 2025 and count < 8) else ""
    print(f"  {year}: {count}{flag}")

Before removal : 235
Removed        : 3
Corpus         : 232

Date range     : 1997-03-25 to 2026-07-29
Joint actions flagged: 4

Statements per year
  1994: 0
  1995: 0
  1996: 0
  1997: 1
  1998: 3
  1999: 6
  2000: 8
  2001: 11
  2002: 8
  2003: 8
  2004: 8
  2005: 8
  2006: 8
  2007: 9
  2008: 9
  2009: 8
  2010: 9
  2011: 8
  2012: 8
  2013: 8
  2014: 8
  2015: 8
  2016: 8
  2017: 8
  2018: 8
  2019: 8
  2020: 11
  2021: 8
  2022: 8
  2023: 8
  2024: 8
  2025: 8
  2026: 5


232 statements, March 1997 to July 2026, with no year below the scheduled
cadence.

The removals confirm themselves in the year counts:

- 2019 went from nine to eight, 2020 from twelve to eleven, 2025 from nine to
  eight.
- Each dropped document came out of a year showing one above the cadence, which
  is what you would expect if they were extra documents rather than statements
  the corpus needed.

Every count now has an explanation from outside the code:

| Years | Count | Why |
| --- | --- | --- |
| 1994 to 1996 | 0 | The FOMC first announced a meeting outcome in February 1994 and did not issue statements routinely |
| 1997 to 1999 | 1, 3, 6 | Statements issued only when policy changed |
| 2000 onward | 8 | Statement after every scheduled meeting, a practice adopted in January 2000 |
| 2001 | 11 | Intermeeting actions, including the response to September 11 |
| 2007, 2008, 2010 | 9 | Intermeeting cuts and coordinated actions |
| 2020 | 11 | The COVID response, including two intermeeting actions in March |
| 2026 | 5 | Partial year, corpus ends July 29 |

That table is the point of the cadence check. Every departure from eight is
explained by something that happened and none by the scraper. When the check
first ran in Section 4.0 it produced five zeros with no such explanation, and
that difference is the whole distinction between a corpus you can use and one
that merely has rows in it.

## 11.0 The Corpus Quality Report

Every check so far has run once, at the point where the thing it tests was built.
That is the wrong place for them to live permanently:

- The Fed will reorganize its site again.
- A future statement will use phrasing that does not match the vote pattern.
- Someone, possibly you, will change a threshold and not notice what it affected
  three sections away.

When that happens, the check that catches it needs to be in one place running
against the finished corpus, not scattered through the notebook attached to the
code it was written for.

So we collect them into a single report that runs every time the corpus is
rebuilt:

- Statement count and date range.
- Statements per year against the meeting cadence.
- Word count by era against known statement lengths.
- Encoding artifacts.
- Voting records split, against the count of statements containing a vote phrase.
- Release times captured.

The report prints numbers rather than passing or failing, with one exception: the
checks with a known correct answer are marked. A report that only ever says OK
trains you not to read it.

What to look for in the output:

- The release time count, the one number in this notebook that is not resolved.

In [16]:
# =============================================================================
# Section 11.0: Corpus quality report
# =============================================================================

# The release time appears in the page as "For release at 2:00 p.m. EDT".
# Extracting it now, before the report, because the report needs to count it
# and it is a field the corpus should carry.
RE_RELEASE = re.compile(
    r"For release at\s+(\d{1,2}:\d{2}\s*[ap]\.?m\.?)\s*([A-Z]{3,4})?",
    re.IGNORECASE,
)


def extract_release_time(body):
    """
    The stated release time, or None where the page does not carry one.

    Point-in-time integrity depends on this: a statement released at 2:00 p.m.
    cannot be traded on at the 9:30 open of the same day. Where the field is
    absent we do not guess, because a guessed timestamp is worse than a
    missing one.
    """
    match = RE_RELEASE.search(body)
    if not match:
        return None
    zone = f" {match.group(2)}" if match.group(2) else ""
    return f"{match.group(1)}{zone}"


df_corpus["release_time"] = df_corpus["body"].apply(extract_release_time)


def quality_report(df):
    """Run every check from this notebook against the finished corpus."""
    print("=" * 60)
    print("CORPUS QUALITY REPORT")
    print("=" * 60)
    print(f"Statements     : {len(df)}")
    print(f"Date range     : {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"Scrape date    : {SCRAPE_DATE} UTC")
    print(f"Script version : {SCRIPT_VERSION}\n")

    # Cadence. Only years fully inside the corpus window are checked, and
    # only from 2000, when the practice of a statement per meeting began.
    print("--- Meeting cadence ---")
    counts = df["date"].dt.year.value_counts().sort_index()
    checkable = counts.loc[2000:df["date"].max().year - 1]
    below = checkable[checkable < 8]
    print(f"Years from 2000 with fewer than eight statements: {len(below)}"
          f"  <-- must be 0")
    if len(below):
        print(below.to_string())

    # Length by era.
    print("\n--- Word count by era ---")
    print(df.groupby("era")["n_words"].agg(["count", "median", "min", "max"])
          .round(0).to_string())

    # Encoding. Both detectors, since Section 7.0 showed the marker check alone
    # understates the damage.
    print("\n--- Encoding ---")
    markers = df["body"].str.contains(MOJIBAKE_MARKERS, regex=True).sum()
    boms = df["body"].str.contains("\ufeff", regex=False).sum()
    print(f"Statements with mojibake markers : {markers}  <-- must be 0")
    print(f"Statements containing a BOM      : {boms}  <-- must be 0")

    # Voting. The ceiling is the count of statements containing a vote phrase
    # at all, not the corpus size, since some statements genuinely have none.
    print("\n--- Voting records ---")
    has_phrase = df["body"].str.contains(RE_VOTE, regex=True).sum()
    split_off = (df["votes"].str.len() > 0).sum()
    leftover = df["statement"].str.contains(RE_VOTE, regex=True).sum()
    print(f"Statements containing a vote phrase : {has_phrase}")
    print(f"Voting records split off            : {split_off}  <-- must equal above")
    print(f"Vote phrase left in statement text  : {leftover}  <-- must be 0")
    print(f"Opening vote tally captured         : {df['tally_for'].notna().sum()}")

    # Release time. Reported rather than checked, because the correct number
    # is not known and the field is genuinely absent from older pages.
    print("\n--- Release time ---")
    captured = df["release_time"].notna().sum()
    print(f"Release time captured : {captured} of {len(df)}")
    print("By era:")
    print(df.groupby("era")["release_time"]
          .agg(["count", "size"]).to_string())

    print("\n--- Flags ---")
    print(f"Joint central bank actions flagged : {df['is_joint_action'].sum()}")
    print("=" * 60)


quality_report(df_corpus)

CORPUS QUALITY REPORT
Statements     : 232
Date range     : 1997-03-25 to 2026-07-29
Scrape date    : 2026-08-22 UTC
Script version : 16.1.0

--- Meeting cadence ---
Years from 2000 with fewer than eight statements: 0  <-- must be 0

--- Word count by era ---
          count  median  min   max
era                               
1 legacy     61   206.0  107   357
2 mid        43   295.0   85  1394
3 modern    128   462.0  145   910

--- Encoding ---
Statements with mojibake markers : 0  <-- must be 0
Statements containing a BOM      : 0  <-- must be 0

--- Voting records ---
Statements containing a vote phrase : 196
Voting records split off            : 196  <-- must equal above
Vote phrase left in statement text  : 0  <-- must be 0
Opening vote tally captured         : 2

--- Release time ---
Release time captured : 90 of 232
By era:
          count  size
era                  
1 legacy      0    61
2 mid         2    43
3 modern     88   128

--- Flags ---
Joint central bank actions fl

Every check with a known correct answer passes, and the corpus holds 232
statements from March 1997 to July 2026.

The release time is the one number that does not resolve, and the era breakdown
explains why:

| Era | Captured | Total |
| --- | --- | --- |
| 1 legacy | 0 | 61 |
| 2 mid | 2 | 43 |
| 3 modern | 88 | 128 |

Pre-2006 pages carry no release time at all, and the practice of stating one was
adopted gradually over the mid era. This is a property of the documents rather
than a failure of the parser, and no amount of pattern work recovers a field that
was never published.

That matters more than it looks:

- A statement is tradable only after it is released, and 40% of this corpus does
  not say when that was.
- For the modern era the scheduled release is 2:00 p.m. Eastern and the captured
  values confirm it, so a same-day trade is possible from the close.
- For the legacy era we do not know, and the correct handling is to trade the
  following day rather than assume.

We record the field where it exists and leave it empty where it does not. Filling
the gap with 2:00 p.m. would produce a corpus where every row has a timestamp and
142 of them are invented, and an invented timestamp that is right most of the
time is worse than an absent one, because nothing downstream can tell them apart.

Notebook 16.2 uses next-day returns throughout, which sidesteps the question
rather than answering it, and states it there as a limitation.

The report itself is the more durable output of this section. Nine checks against
the finished corpus, printing numbers rather than a verdict. When the Fed changes
its site again and this notebook is re-run, the report tells you what broke
without anyone remembering which section the relevant check lived in.

## 12.0 Writing the Corpus

The corpus is finished and needs to leave this notebook as a file, because
Notebook 16.2 loads that file and never touches the network.

That separation is a working practice rather than a convenience:

- Acquisition and analysis are different jobs with different failure modes.
- Joining them means every analysis run depends on a website behaving the way it
  did last time.
- The file is the boundary: 16.1 can break without 16.2 noticing, and 16.2 can
  re-run a hundred times without sending a request to the Fed.

The columns we write, and why each is there:

- `date`, `url`, `era`: identity and provenance for each row.
- `statement`: the text with the voting record removed, which is what the
  language features in 16.2 are computed from.
- `votes`: the voting record, kept separate so names do not contaminate a measure
  of policy language.
- `tally_for`, `tally_against`: the opening vote tally where present.
- `release_time`: where the page states one, empty where it does not.
- `n_words`: the length check, carried so the report runs without recomputing.
- `is_joint_action`: the flag from Section 10.0.
- `scraped_at`, `script_version`: when this corpus was built and by what.

The last two are most often left out and most needed later. A corpus with no
build date cannot be reconciled against a rebuild, and the first question anyone
asks about a result that will not reproduce is which version of the data produced
it.

What to look for in the output:

- The file size and row count.
- The reload check, which reads the file back and confirms it survived the round
  trip.

In [17]:
# =============================================================================
# Section 12.0: Writing the corpus to disk
# =============================================================================

OUTPUT_COLUMNS = [
    "date", "url", "era",
    "statement", "votes",
    "tally_for", "tally_against",
    "release_time", "n_words",
    "is_joint_action",
    "scraped_at", "script_version",
]

df_out = df_corpus.copy()
df_out["scraped_at"] = SCRAPE_DATE
df_out["script_version"] = SCRIPT_VERSION
df_out = df_out[OUTPUT_COLUMNS].sort_values("date").reset_index(drop=True)

# Statement text contains commas, quotes, and newlines. CSV handles all three
# through quoting, and pandas does it correctly by default, but the round trip
# is the only way to know it worked on this particular text.
df_out.to_csv(OUTPUT_CSV, index=False)

size_kb = OUTPUT_CSV.stat().st_size / 1024
print(f"Written: {OUTPUT_CSV}")
print(f"Rows   : {len(df_out)}")
print(f"Size   : {size_kb:,.0f} KB\n")

# --- Read it back ------------------------------------------------------------
# A file that writes without error and reads back wrong is a real failure mode
# with text data, and it surfaces two notebooks later as a statement that ends
# mid-sentence. Checking here costs one line.
df_check = pd.read_csv(OUTPUT_CSV, parse_dates=["date"])

print("Round trip check:")
print(f"  Rows match          : {len(df_check) == len(df_out)}  <-- must be True")
print(f"  Columns match       : {list(df_check.columns) == OUTPUT_COLUMNS}  <-- must be True")

# Compare the text itself rather than just the shape. Total characters across
# the statement column catches truncation that a row count would not.
chars_before = df_out["statement"].str.len().sum()
chars_after = df_check["statement"].str.len().sum()
print(f"  Statement characters: {chars_before:,} written, {chars_after:,} read back")
print(f"  Text intact         : {chars_before == chars_after}  <-- must be True")

print("\nFirst three rows:")
print(df_check[["date", "era", "n_words", "release_time"]].head(3).to_string(index=False))
print("\nLast three rows:")
print(df_check[["date", "era", "n_words", "release_time"]].tail(3).to_string(index=False))

Written: fomc_cache/fomc_statements.csv
Rows   : 232
Size   : 621 KB

Round trip check:
  Rows match          : True  <-- must be True
  Columns match       : True  <-- must be True
  Statement characters: 510,313 written, 510,313 read back
  Text intact         : True  <-- must be True

First three rows:
      date      era  n_words release_time
1997-03-25 1 legacy      142          NaN
1998-09-29 1 legacy      107          NaN
1998-10-15 1 legacy      161          NaN

Last three rows:
      date      era  n_words  release_time
2026-04-29 3 modern      367 2:00 p.m. EDT
2026-06-17 3 modern      157 2:00 p.m. EDT
2026-07-29 3 modern      193 2:00 p.m. EDT


232 rows, 621 KB, and the text survives the round trip with every character
accounted for.

The character comparison is the check worth keeping. A row count confirms the
right number of records and says nothing about whether the text inside them is
complete, and truncated statement text is the kind of problem that surfaces two
notebooks later as a language measure behaving oddly in one era.

The first and last rows serve as a final sanity check on the two ends of the
corpus:

- March 1997 at 142 words with no release time.
- July 2026 at 193 words released at 2:00 p.m. Eastern.

Both are what those documents actually look like.

This file is the artifact. Notebook 16.2 loads it and works entirely from it,
which means the analysis in that notebook is reproducible regardless of what
happens to the Federal Reserve's website.

## 13.0 Summary

The output is one file of 232 FOMC policy statements spanning thirty years,
with the voting record separated from the statement text and provenance stamped
on every row.

The work to get there ran to four failures, and the shape of them is the part
worth carrying into your own projects:

| Failure | What the output looked like | What caught it |
| --- | --- | --- |
| Two URL schemes assumed, three exist | 192 plausible statements | Meeting cadence, from outside the code |
| Server declared the wrong character set | Text that read as almost correct | Decoding twice and comparing |
| Paragraph extraction with no container | Statements five times their real length | Known document length |
| Vote marker matched literal phrasings | A research finding about a new Chair | Reading the document |

None of the four raised an error. Every one of them produced output that a
reasonable person would accept, and three of the four produced output I did
accept until an external number disagreed with it.

That is the argument this notebook exists to make. Text pipelines do not fail
loudly, and the checks that catch them cannot be derived from the code, because
code that is wrong in a consistent way will validate itself. Eight scheduled
meetings a year, two hundred words per statement, the Committee records its
votes: none of those facts is in the HTML and all three found something.

The fourth failure is the one I would ask you to remember, because it is the
only one where the data was correct. A `str.find` returned -1, I read it as a
fact about the Federal Reserve rather than a fact about my search string, and it
became a structural break I was building on. The distance between a parser bug
and a research finding is one interpretation, and the check that closes it is
opening the source document before a null result becomes a conclusion.

Notebook 16.2 takes this corpus and asks whether anything in it predicts market
returns. The answer matters less than the process, and this chapter's version of
that process starts from a corpus we have reason to trust.